# Module 3 — Where Data Lives

*Live session, 1.5h.*

**Goal.** The same facts, put into six real database engines, and the same
questions asked of all six. Not to crown one — the winner changes with the
question, and learning to read that is the point.

**The data.** Module 2 modelled two of Chicago's open datasets — the **contracts**
and the **business licences** — on 46 contract rows and 54 licences. This is the
same two files at full scale: **2,137,944 nodes and 4,073,458 edges**, with the
entity layer that decides which rows describe the same company. You are given it
already loaded.

The sample we read line by line in this notebook is **six nodes and six edges** —
small enough to translate by hand, and chosen because it contains one company that
appears in both files and was resolved into one, and one that was not.

**The tooling.** Module 2 ended with *no server to start, no database to install*.
Everything ran in the notebook, and that was the right place to learn what a model
is.

This week is the opposite, and the difference is the subject. Four of the six
engines are servers in containers, two are libraries that arrive with `pip
install` and die when the kernel stops — and today **all six are running**, so the
questions get asked live rather than read.

The four servers share one Docker VM, so every latency below was paid under
contention. It matters less than you would guess, and *The same question, six
times* says by how much.

## The trip is not free

In module 2 you wrote the same three facts three ways — OWL, SHACL, a property
graph — and all three fitted in memory. Nothing was measured, because nothing
needed to be: 46 contract rows and 54 licences cost nothing to say twice.

At 2.1 million nodes the second saying has a price. It is smaller than you would
guess, and it is concentrated in two places — which is the useful part.

**We start with one contract.** Chicago's Department of Planning and Development
holds contract **275989**, *The Bloc; ARP-CDG*, running from 11 June 2024 to 1
July 2025. Over that period it had **two vendors**: one from 10 July 2024 until 8
May 2025, and a second from 8 May 2025 onward. Both trade as *The Bloc*, both at
1345 N Karlov Ave.

Ask the graph *who was the vendor in March 2025* and you get the first; ask for
June and you get the second. Nothing on either company says so, and nothing on
the contract says so. **The dates are on the edge**, and that is the only place
they can go — a validity period is not a fact about the contract, nor about the
company, but about the award that joins them for a while.

The next cell takes that contract — nodes, properties, edges and all — and says
the same thing in RDF, counting both sides.

In [1]:
# Contract 275989 as the graph holds it, and the same facts as triples.
# These are the real properties and the real edges, copied out of the loaded
# graph once and FIXED here on purpose: the six nodes have to stay the same
# all term, so the translation can be checked by hand against this page.
#
# Two organisations are called "The Bloc" and one of them came from the LICENCES
# while the other two came from the CONTRACTS. That is module 2's output sitting
# in the data, and it is why this subgraph is the one worth writing out.
NODES = {
    "Contract/275989": {
        "endDate":   ["2025-07-01"],
        "id":        ["275989"],
        "name":      ["The Bloc; ARP-CDG"],
        "names":     ["The Bloc; ARP-CDG"],
        "startDate": ["2024-06-11"],
    },
    # six of this record's eighteen properties, to keep the subgraph readable
    "LicenseRecord/1292422": {
        "businessName": ["THE BLOC"],
        "endDate":      ["2027-04-15"],
        "id":           ["1292422"],
        "startDate":    ["2025-04-09"],
        "status":       ["AAI"],
        "code":         ["1023"],
    },
    # from the CONTRACTS, and resolved: it has a Louvain community
    "Organization/contracts:10598723": {
        "addressCities":      ["Chicago"],
        "addressPostalCodes": ["60651"],
        "addressStates":      ["IL"],
        "addresses":          ["1345 N Karlov Ave"],
        "clusterKey":         ["L11595"],
        "id":                 ["contracts:10598723"],
        "louvainCoarsest":    ["11595"],
        "louvainFinest":      ["11595"],
        "louvainLevels":      ["11595", "11595", "11595"],   # <- three levels
        "name":               ["The Bloc"],
        "names":              ["The Bloc"],
        "source":             ["CONTRACTS"],
    },
    # from the LICENCES. Same address in capitals, same community, and it is the
    # one that carries the licence.
    "Organization/licenses:514614": {
        "addressCities":      ["CHICAGO"],
        "addressPostalCodes": ["60651"],
        "addressStates":      ["IL"],
        "addresses":          ["1345 N KARLOV AVE"],
        "clusterKey":         ["L11595"],
        "id":                 ["licenses:514614"],
        "louvainCoarsest":    ["11595"],
        "louvainFinest":      ["11595"],
        "louvainLevels":      ["11595", "11595", "11595"],
        "name":               ["THE BLOC"],
        "names":              ["THE BLOC"],
        "source":             ["LICENSES"],
    },
    
    # the SECOND vendor of the same contract. Also called The Bloc, also at
    # 60651 -- and NOT resolved: no Louvain properties, because nothing was
    # found similar enough to it.
    "Organization/contracts:105504356L": {
        "addressCities":      ["CHICAGO"],
        "addressPostalCodes": ["60651"],
        "addressStates":      ["IL"],
        "clusterKey":         ["contracts:105504356L"],
        "id":                 ["contracts:105504356L"],
        "name":               ["The Bloc"],
        "names":              ["The Bloc"],
        "source":             ["CONTRACTS"],
    },
    # the entity module 2 minted. It names both of its members.
    "OrganizationGroup/L11595": {
        "clusterId": ["L11595"],
        "ids":       ["contracts:10598723", "licenses:514614"],
        "name":      ["The Bloc"],
        "names":     ["The Bloc", "THE BLOC"],
        "sources":   ["CONTRACTS", "LICENSES"],
    },
}
# (source, type, target, properties on the edge)
EDGES = [
    ("Contract/275989", "HAS_VENDOR", "Organization/contracts:10598723",
     {"validFrom": "2024-07-10", "validTo": "2025-05-08"}),
    ("Contract/275989", "HAS_VENDOR", "Organization/contracts:105504356L",
     {"validFrom": "2025-05-08"}),
    # the edge that says WHY the two are one company. Module 2 wrote it.
    ("Organization/licenses:514614", "IS_SIMILAR_TO",
     "Organization/contracts:10598723",
     {"method": "NORM_ADDRESS", "methods": "NORM_ADDRESS|SAME_ZIP", "score": "1.0"}),
    ("Organization/licenses:514614", "ORG_HAS_LICENSE",
     "LicenseRecord/1292422", {}),
    ("Organization/contracts:10598723", "BELONGS_TO_ORG_GROUP",
     "OrganizationGroup/L11595", {}),
    ("Organization/licenses:514614", "BELONGS_TO_ORG_GROUP",
     "OrganizationGroup/L11595", {}),
]

# --- THE PICTURE FIRST. One edge, two properties, and the same fact in RDF.
#     Drawn from EDGES[0] above, so the drawing cannot drift from the data.
_src, _rel, _dst, _p = EDGES[0]
_vendor = _dst.split("/", 1)[1]
print(f"""ONE EDGE, TWO PROPERTIES -- the first of contract 275989's two vendors.

   BEFORE — the property graph. The dates live ON the edge:

                                   ┌ validFrom  {_p['validFrom']} ┐
      (Contract 275989) ───[:{_rel}                    ]───> ({_vendor})
                                   └ validTo    {_p['validTo']} ┘

   AFTER — RDF. A triple has three slots, all three are taken, and there is
   nowhere to put a date. So a NODE is minted to stand for the edge:

      (id:Contract/275989) ────── 1. voc:{_rel} ──────> (id:Organization/…)
               ▲                                                     ▲
               │ 3. voc:subject                      4. voc:object   │
               │       ┌──────────────────────────────┐              │
               └───────┤   id:stmt/{_rel}/0       ├──────────────┘
                       └──────────────┬───────────────┘
                                      │
                 2. rdf:type ─────────┼──> voc:{_rel}Statement
                 5. voc:validFrom ────┼──> "{_p['validFrom']}"^^xsd:date
                 6. voc:validTo ──────┴──> "{_p['validTo']}"^^xsd:date

   SIX triples where the property graph had one edge. Count them by what they do:

      1 is the edge itself, kept so an ordinary query still works
      2, 3, 4 are the MACHINERY of the node that replaces it
      5, 6 are the dates — the only two you actually wanted to store

   The machinery is the cost: rdf:type, voc:subject, voc:object. THREE TRIPLES
   PER EDGE, every time, whatever the properties are.
""")

# --- Now the whole subgraph, counted. One VALUE is one assertion, because that
#     is the only way to compare with RDF, where one value is one triple.
nvals = sum(len(v) for p in NODES.values() for v in p.values())
nprops = sum(len(p) for p in NODES.values())
evals = sum(len(p) for _, _, _, p in EDGES)
lpg = len(NODES) + nvals + len(EDGES) + evals

print("What this subgraph asserts")
for n, what, eg in [
        (len(NODES), "the nodes exist", "(Organization contracts:10598723)"),
        (nvals, "their property values", f'(name = "The Bloc", one of {nprops})'),
        (len(EDGES), "the edges exist", "(:IS_SIMILAR_TO, licences -> contracts)"),
        (evals, "the property values ON those edges", "(score = 1.0, on that edge)")]:
    print(f"   {n:>3}   {what:<36}{eg}")

# --- the same facts as triples, built the way the real export builds them.
#     The rules are the exporter's -- the one that produced the payload you
#     downloaded: the statement-node shape, the typed literals and the
#     opaque `stmt/` URI are all its, rewritten here so the page can be
#     read on its own. Module 4 is where that exporter gets written.
P, V = "http://kgtp.org/id/", "http://kgtp.org/voc/"
RDF = "http://www.w3.org/1999/02/22-rdf-syntax-ns#"
XSD = "http://www.w3.org/2001/XMLSchema#"
INT = {"louvainCoarsest", "louvainFinest", "louvainLevels"}
DATE = {"startDate", "endDate", "validFrom", "validTo"}

def uri(ref):
    lab, key = ref.split("/", 1)
    return f"<{P}{lab}/{key.replace(':', '%3A')}>"

def lit(p, v):
    if p in INT:  return f'"{v}"^^<{XSD}integer>'
    if p in DATE: return f'"{v}"^^<{XSD}date>'
    return f'"{v}"'

nt = []
for ref, props in NODES.items():
    u, lab = uri(ref), ref.split("/", 1)[0]
    nt.append(f'{u} <{RDF}type> <{V}{lab}> .')
    for p, vals in props.items():
        nt += [f'{u} <{V}{p}> {lit(p, x)} .' for x in vals]
stmt = 0
for a, rel, b, props in EDGES:
    ua, ub = uri(a), uri(b)
    nt.append(f'{ua} <{V}{rel}> {ub} .')
    if props:                               # only then is a statement node made
        us = f'<{P}stmt/{rel}/{stmt}>'; stmt += 1
        nt += [f'{us} <{RDF}type> <{V}{rel}Statement> .',
               f'{us} <{V}subject> {ua} .',
               f'{us} <{V}object> {ub} .']
        nt += [f'{us} <{V}{p}> {lit(p, v)} .' for p, v in props.items()]

import pyoxigraph as ox
store = ox.Store()
store.load("\n".join(nt).encode(), ox.RdfFormat.N_TRIPLES)

print(f"\n{'the property graph, asserting all of that':<44}{lpg:>5}")
print(f"{'the same facts written as triples':<44}{len(nt):>5}   {len(nt)-lpg:+d}"
      f"   = 3 machinery triples x {stmt} edges")
print(f"{'what the store actually kept':<44}{len(store):>5}   {len(store)-len(nt):+d}")

print(f"""
Two things happened, and only one of them is a cost.

The {len(nt)-lpg:+d} is the PRICE of the three edges that carry properties -- 2, 3 and 4 of
the picture, three times over -- and it buys something: ask who the vendor was
in March 2025 and the store can still answer, through the statement node.

The {len(store)-len(nt):+d} is a LOSS. RDF is a set, so three identical values collapse into one.
The next cell asks the store how many Louvain levels it thinks there were.""")


ONE EDGE, TWO PROPERTIES -- the first of contract 275989's two vendors.

   BEFORE — the property graph. The dates live ON the edge:

                                   ┌ validFrom  2024-07-10 ┐
      (Contract 275989) ───[:HAS_VENDOR                    ]───> (contracts:10598723)
                                   └ validTo    2025-05-08 ┘

   AFTER — RDF. A triple has three slots, all three are taken, and there is
   nowhere to put a date. So a NODE is minted to stand for the edge:

      (id:Contract/275989) ────── 1. voc:HAS_VENDOR ──────> (id:Organization/…)
               ▲                                                     ▲
               │ 3. voc:subject                      4. voc:object   │
               │       ┌──────────────────────────────┐              │
               └───────┤   id:stmt/HAS_VENDOR/0       ├──────────────┘
                       └──────────────┬───────────────┘
                                      │
                 2. rdf:type ─────────┼──> voc:HAS_

### The cost, and the two places it sits

Nine triples more on three edges, and four lost. The whole graph has one number
for the same toll, and it is at the end of this cell. But before the table, one
thing about its shape, because the shape misleads.

**RDF has no nodes.** It has subjects, and what makes a subject *exist* is a
triple — `<thing> rdf:type <Class>`. So the first row below is not "both sides have
2.1 million nodes": it is "the property graph has 2,137,944 nodes, and saying so in
RDF takes 2,137,944 triples". Same count, different kind of thing, and the table
can only compare them by counting **assertions on one side against triples on the
other**.

That is the convention this whole section runs on, and it is worth saying once, plainly:

| in the property graph | counted as | in RDF |
|---|---|---|
| a node exists | 1 | one `rdf:type` triple |
| a property with one value | 1 | one triple |
| a property with a list of 3 values | **3** | three triples |
| an edge with no properties | 1 | one triple |
| an edge with 2 properties | **3** | **six** triples — the picture above |

Without that convention there is nothing to compare: *2.1 million nodes* and
*36.8 million triples* are not two measurements of the same thing.

With it, over the whole graph:

| | property graph | RDF |
|---|---:|---:|
| nodes exist / `rdf:type` triples | 2,137,944 | 2,137,944 |
| property values on nodes — one value, one assertion | 28,737,228 | 28,737,228 |
| edges with no properties | 3,600,765 | 3,600,765 |
| **subtotal — all of it 1 to 1** | **34,475,937** | **34,475,937** |
| the edges that carry properties — picture **1** | 472,693 | 472,693 |
| their property values — picture **5, 6** | 514,267 | 514,267 |
| the statement nodes — picture **2, 3, 4** | — | **1,418,079** |
| **total written** | **35,462,897** | **36,880,976** |

One row of that table is not 1 to 1, and it is the one the picture called
machinery: **1,418,079 = 3 × 472,693**. Three triples for every edge that carries
properties, and the count does not depend on how many properties it carries —
one or ten, the statement node costs the same three. That is the fact about RDF.

**On this graph it comes to 1.04×**, and that figure is about Chicago rather than
about RDF: 34,475,937 of 35,462,897 assertions — 97% — cross one to one, because
only 11.6% of the edges carry properties at all. What moves the 1.04 is that
share, not the number of properties on an edge.


In [2]:
# One list of three values, modelled three ways. All three are legal RDF.
#
# The value is real: Organization contracts:10598723 has louvainLevels
# [11595, 11595, 11595] -- the Louvain hierarchy has three levels and this company
# stayed in the same community at every one.
import json, os
import pyoxigraph as ox

S = "<http://kgtp.org/id/Organization/contracts%3A10598723>"
V, R = "http://kgtp.org/voc/", "http://www.w3.org/1999/02/22-rdf-syntax-ns#"
X = "http://www.w3.org/2001/XMLSchema#"
LEVELS = ["11595", "11595", "11595"]
SHORT = lambda t: (t.replace(S, "id:org").replace(V, "voc:")
                    .replace(R, "rdf:").replace(X, "xsd:").strip())

# ---- A: one triple per element, on the same predicate. What this module did.
A = [f'{S} <{V}louvainLevels> "{g}"^^<{X}integer> .' for g in LEVELS]
A_Q = f'SELECT (COUNT(*) AS ?n) WHERE {{ {S} <{V}louvainLevels> ?g }}'

# ---- B: rdf:List. Ordered, faithful, a chain of blank nodes.
B = [f'{S} <{V}louvainLevels> _:n0 .']
for i, g in enumerate(LEVELS):
    B.append(f'_:n{i} <{R}first> "{g}"^^<{X}integer> .')
    B.append(f'_:n{i} <{R}rest> '
             + (f'_:n{i+1} .' if i < len(LEVELS) - 1 else f'<{R}nil> .'))
B_Q = (f'SELECT (COUNT(*) AS ?n) WHERE {{ {S} <{V}louvainLevels> ?h . '
       f'?h <{R}rest>* ?c . ?c <{R}first> ?g }}')

# ---- C: a node per level, carrying WHICH level it was.
C = []
for i, g in enumerate(LEVELS):
    C += [f'{S} <{V}louvainLevel> _:l{i} .',
          f'_:l{i} <{V}level> "{i}"^^<{X}integer> .',
          f'_:l{i} <{V}community> "{g}"^^<{X}integer> .']
C_Q = (f'SELECT (COUNT(*) AS ?n) WHERE {{ {S} <{V}louvainLevel> ?l . '
       f'?l <{V}level> ?i ; <{V}community> ?g }}')

print("""`_:x` below is a BLANK NODE: a node with no URI, which exists only inside this
file. B and C both mint them -- B chains them, C keeps them flat.
""")

for name, triples, q, note in [
        ("A — repeat the predicate", A, A_Q,
         "the obvious translation, and what this module's export does"),
        ("B — rdf:List, a CHAIN of blank nodes", B, B_Q,
         "the standard way to say 'an ordered list'"),
        ("C — one blank node per level, carrying its index", C, C_Q,
         "say what each element MEANS, not just its value")]:
    print("=" * 74)
    print(f"{name}   ({note})")
    print("=" * 74)
    print("   what gets written:")
    for t in triples:
        print(f"      {SHORT(t)}")
    st = ox.Store()
    st.load("\n".join(triples).encode(), ox.RdfFormat.N_TRIPLES)
    print(f"\n   {len(triples)} triples written, {len(st)} IN THE STORE"
          + (f"   <-- {len(triples)-len(st)} DISAPPEARED"
             if len(st) < len(triples) else ""))
    print(f"\n   the question: how many Louvain levels does this company have?")
    print(f"      {SHORT(q)}")
    print(f"   the answer:  {[t.value for row in st.query(q) for t in row][0]}"
          f"       (the true answer is 3)\n")

print("""======================================================================
A does not approximate the list. A ANSWERS WRONG.
======================================================================

   RDF is a SET           three identical triples are ONE triple
   so A writes 3          and the store keeps 1
   the question           "how many levels?" comes back 1 -- the true answer is 3
   and nothing failed     correct SPARQL, correct store, false answer, no warning

   a list is a LIST       in the property graph a repeat is information:
                          [11595, 11595, 11595] says the hierarchy found this
                          company in the same community at all three levels

   B answers 3            7 triples, and a rdf:rest* path to read it back
   C answers 3            9 triples, a plain join, and the only one of the three
                          that can say WHICH level, because it wrote the index

   THE LOSS IS NOT A PROPERTY OF RDF. RDF has rdf:List, and C is simpler still.
   It is a property of the MAPPING WE CHOSE.
""")

# The bill. One of its three numbers is not written here:
#
#   - the count IN THE STORE is read from the loaders' own record, one entry per
#     triple store, and the three are compared rather than asserted to agree
#
# Only the written-out total stays a constant: counting it means re-reading
# kgtp.nt.gz, 288 MB compressed, which is not a thing to do in a lesson.
WRITTEN = 36_880_976
TIMES = json.load(open(os.path.join("import",
                                    "load-times.json")))
stored = {e: v["triples"] for e, v in TIMES.items() if "triples" in v}
assert len(set(stored.values())) == 1, stored      # or the sentence below is false
in_store = next(iter(stored.values()))


print(f"""
THE BILL, measured rather than estimated

   {WRITTEN:>10,}   triples written out by the exporter
   {in_store:>10,}   in the store after loading, and all {len(stored)} triple stores agree
                ({', '.join(sorted(stored))})
   {WRITTEN-in_store:>10,}   collapsed -- every one of them a voc:louvainLevels, and nothing
                else in this graph has a list that repeats a value

   Mapping C would have cost about 1.6 million triples, 4% of the file.""")


`_:x` below is a BLANK NODE: a node with no URI, which exists only inside this
file. B and C both mint them -- B chains them, C keeps them flat.

A — repeat the predicate   (the obvious translation, and what this module's export does)
   what gets written:
      id:org <voc:louvainLevels> "11595"^^<xsd:integer> .
      id:org <voc:louvainLevels> "11595"^^<xsd:integer> .
      id:org <voc:louvainLevels> "11595"^^<xsd:integer> .

   3 triples written, 1 IN THE STORE   <-- 2 DISAPPEARED

   the question: how many Louvain levels does this company have?
      SELECT (COUNT(*) AS ?n) WHERE { id:org <voc:louvainLevels> ?g }
   the answer:  1       (the true answer is 3)

B — rdf:List, a CHAIN of blank nodes   (the standard way to say 'an ordered list')
   what gets written:
      id:org <voc:louvainLevels> _:n0 .
      _:n0 <rdf:first> "11595"^^<xsd:integer> .
      _:n0 <rdf:rest> _:n1 .
      _:n1 <rdf:first> "11595"^^<xsd:integer> .
      _:n1 <rdf:rest> _:n2 .
      _:n2 <rdf:first> "1159

## The six engines

Three property graphs and three triple stores, and **one of each is a library
rather than a server**. That symmetry is the shape of the week.

| | paradigm | how you get it | |
|---|---|---|:-:|
| **Neo4j** | LPG | container | served |
| **FalkorDB** | LPG | container | served |
| **Kuzu** | LPG | `pip install kuzu` | **embedded** |
| **GraphDB** | RDF | container | served |
| **Fuseki** | RDF | container | served |
| **Oxigraph** | RDF | `pip install pyoxigraph` | **embedded** |

**The two embedded ones are not embedded for the same reason.** Kuzu is embedded
*by nature*: a Kuzu server does not exist. Oxigraph is embedded *by choice* —
`pyoxigraph` and the `oxigraph/oxigraph` container are the same engine at the same
version, reading the same files, so running the container would measure a socket
rather than an engine.

In [3]:
# The six are up, and there is ONE way to ask them. Nothing here is counted --
# the cell after next does the counting. This is the INTERFACE, and the two
# languages side by side. The fact asked for is the one the frozen subgraph
# above writes out by hand, so the answer can be recognised instead of trusted.
import os, sys
sys.path.insert(0, os.path.join("..", "..", "tools", "03-where-data-lives"))
from engines import ask, LPG, RDF, EMBEDDED, GRAPH

ID = "contracts:10598723"
URI = "http://kgtp.org/id/Organization/contracts%3A10598723"
NAME = "http://kgtp.org/voc/name"
CYPHER = f"MATCH (o:Organization {{id:'{ID}'}}) RETURN o.name AS name"
SPARQL = (f"SELECT ?name WHERE {{ GRAPH <{GRAPH}> {{\n"
          f"          <{URI}> <{NAME}> ?name }} }}")
# The same question with the GRAPH clause taken off. Not a slip: it is a
# difference between the three stores that the one interface hides.
BARE = f"SELECT ?name WHERE {{ <{URI}> <{NAME}> ?name }}"

# What is printed is what is SENT, with the two vocabulary prefixes shortened.
# The graph's name is a datum, so it stays whole.
short = lambda q: (q.replace("http://kgtp.org/id/", "id:")
                    .replace("http://kgtp.org/voc/", "voc:"))

def answer(engine, query):
    try:
        rows = ask(engine, query)
        return rows[0][0] if rows else "(no rows)"
    except Exception as ex:            # a store that is down names itself here
        return f"not answering -- {type(ex).__name__}"

print(f"""THE SAME FACT, ASKED OF ALL SIX. Organization {ID} -- the company the
opening cells write out by hand -- what is its name?

   the three property graphs, in Cypher:
      {CYPHER}

   the three triple stores, in SPARQL:
      {short(SPARQL)}
""")

for e in LPG + RDF:
    lang = "Cypher" if e in LPG else "SPARQL"
    where = "embedded" if e in EMBEDDED else "served"
    print(f"   {e:<10}{lang:<9}{where:<11}{answer(e, CYPHER if e in LPG else SPARQL)}")

print(f"""
Six engines, two languages, one call: ask(engine, query). Behind it a bolt
driver, a Redis connection, an in-process Kuzu, two HTTP endpoints and an
in-process store -- and every one of them returns the same shape, rows of lists.

AND THAT GRAPH CLAUSE IS NOT DECORATION. The same question without it:

      {short(BARE)}
""")
for e in RDF:
    a = answer(e, BARE)
    print((f"   {e:<10}{a:<12}"
           + ("   <-- named graphs are not in its default graph"
              if a == "(no rows)" else "")).rstrip())
print("""
Correct SPARQL, no error, and nothing came back. Name the graph.""")


THE SAME FACT, ASKED OF ALL SIX. Organization contracts:10598723 -- the company the
opening cells write out by hand -- what is its name?

   the three property graphs, in Cypher:
      MATCH (o:Organization {id:'contracts:10598723'}) RETURN o.name AS name

   the three triple stores, in SPARQL:
      SELECT ?name WHERE { GRAPH <http://kgtp.org/graph/kgtp> {
          <id:Organization/contracts%3A10598723> <voc:name> ?name } }



   neo4j     Cypher   served     The Bloc
   falkordb  Cypher   served     The Bloc


   kuzu      Cypher   embedded   The Bloc
   graphdb   SPARQL   served     The Bloc
   fuseki    SPARQL   served     The Bloc
   oxigraph  SPARQL   embedded   The Bloc

Six engines, two languages, one call: ask(engine, query). Behind it a bolt
driver, a Redis connection, an in-process Kuzu, two HTTP endpoints and an
in-process store -- and every one of them returns the same shape, rows of lists.

AND THAT GRAPH CLAUSE IS NOT DECORATION. The same question without it:

      SELECT ?name WHERE { <id:Organization/contracts%3A10598723> <voc:name> ?name }

   graphdb   The Bloc
   fuseki    The Bloc
   oxigraph  (no rows)      <-- named graphs are not in its default graph

Correct SPARQL, no error, and nothing came back. Name the graph.


### How the data got in

Before comparing six engines you owe one check: **do they hold the same facts?**
Not approximately — exactly. A benchmark over six stores with different contents is
six benchmarks. The cell after next asks all six.

The load times that follow are **every engine on its own offline importer** — one
condition, not six. Two words have to stay apart to read them:

- **batched** — thousands of rows per statement instead of one. Every path here
  does this, and the next cell measures what one-at-a-time would have cost.
- **offline importer** — the engine's own loader, writing the store directly with
  the server down, or in-process where there is no server.

| | the tool | what it is |
|---|---|---|
| **Neo4j** | `neo4j-admin database import full` | offline, server down |
| **Kuzu** | `COPY FROM` an Arrow table | in-process |
| **FalkorDB** | `falkordb-bulk-insert` | an **external** package — the image ships no loader, and it has **no date type**: every date lands as a string |
| **Oxigraph** | `Store.bulk_load` | in-process |
| **GraphDB** | `importrdf preload` | offline, server down |
| **Fuseki** | `tdb2.xloader` | offline, server down |

Two of the six can also be loaded through a client, and on the machine this module
was built on that priced the tool with the engine held fixed. Neo4j: **13.2 s** with
its own importer, 147.5 s through a client — eleven times. FalkorDB: **33.6 s**
against 147.8 s — four times. Those two client figures are a record: this
repository ships only the importers.

The FalkorDB row has a price that the clock does not show. A date that arrives as
a string still prints like a date, so `WHERE n.startDate > date('2020-01-01')`
answers **0** on FalkorDB, with no error, where Neo4j and Kuzu answer thousands.
None of this lesson's questions to FalkorDB compares a date; yours might.

In [4]:
# Why every loader in this repository writes in batches: the two shapes, and
# the same N rows each way on all six engines.
#
# READ from the record rather than re-taken here, for the same reason as the
# LOAD column. It was taken once, one engine at a time, with N = 2,000: it costs
# 66 s and needs the write lock on the two embedded stores, which the rest of
# this notebook holds open. At a lesson-sized N the ratios stop separating -- re-run four
# times on 300 rows (13/10/2026) FalkorDB came out between 4x and 26x, which
# puts a served engine inside the embedded band and makes the table disagree
# with the paragraph under it.
import json, os

REC = json.load(open(os.path.join("import",
                                  "batching.json")))
N, RESULTS = REC["rows"], REC["results"]
EMBEDDED = ("kuzu", "oxigraph")

print(f"""ONE ROW AT A TIME, THEN {N:,} ROWS IN ONE STATEMENT. The two shapes:

   a property graph     CREATE (:BatchTest {{i:$i}})                      x {N:,}
                        UNWIND $rows AS r CREATE (:BatchTest {{i:r.i}})    once

   a triple store       INSERT DATA {{ GRAPH <g> {{ <x/0> <x/i> "0" }} }}    x {N:,}
                        INSERT DATA {{ GRAPH <g> {{ ...{N:,} triples... }} }}  once

Written to a throwaway label and a throwaway graph; the loaded payload is never
touched. One engine at a time, so no figure is a share of a contended laptop.
""")

print(f"{'':<13}{'one at a time':>13}{'one batch':>12}{'ratio':>9}")
for name, r in sorted(RESULTS.items(), key=lambda kv: -kv[1]["ratio"]):
    tag = "   (embedded)" if name in EMBEDDED else ""
    print(f"   {name:<10}{r['one_at_a_time']:>10.2f} s{r['one_batch']:>10.3f} s"
          f"{r['ratio']:>8.0f}x{tag}")

print(f"""
Three bands, and each one has its own cause.

   the two EMBEDDED       no round trip at all, so their gap is {N:,} statements
                          parsed and planned instead of one. They are the
                          controls, one per paradigm: whatever the four served
                          engines spend above them is protocol, not engine
   the two served LPG     a row at a time is a ROUND TRIP at a time, over Bolt
                          or over Redis
   the two served RDF     a per-triple INSERT DATA is also a TRANSACTION at a
                          time, so these two pay the trip AND a commit {N:,}
                          times over, which is why they sit an order of
                          magnitude above the rest""")


ONE ROW AT A TIME, THEN 2,000 ROWS IN ONE STATEMENT. The two shapes:

   a property graph     CREATE (:BatchTest {i:$i})                      x 2,000
                        UNWIND $rows AS r CREATE (:BatchTest {i:r.i})    once

   a triple store       INSERT DATA { GRAPH <g> { <x/0> <x/i> "0" } }    x 2,000
                        INSERT DATA { GRAPH <g> { ...2,000 triples... } }  once

Written to a throwaway label and a throwaway graph; the loaded payload is never
touched. One engine at a time, so no figure is a share of a contended laptop.

             one at a time   one batch    ratio
   graphdb        28.13 s     0.038 s     738x
   fuseki         34.73 s     0.093 s     374x
   neo4j           2.32 s     0.057 s      41x
   falkordb        0.47 s     0.013 s      35x
   kuzu            0.15 s     0.011 s      14x   (embedded)
   oxigraph        0.02 s     0.008 s       3x   (embedded)

Three bands, and each one has its own cause.

   the two EMBEDDED       no round trip at all,

In [5]:
# Do the six hold the same facts? Ask them, now. Nothing below is quoted: the
# first engine that answers becomes the reference and the others are compared
# with it.
import os, sys
sys.path.insert(0, os.path.join("..", "..", "tools", "03-where-data-lives"))
from engines import ask, one, up, LPG, RDF, EMBEDDED, GRAPH

down = {e: why for e, why in up().items() if why}
for e, why in down.items():
    print(f"   {e} is not answering -- {why}")

N = "MATCH (n) RETURN count(n) AS n"
E = "MATCH ()-[r]->() RETURN count(r) AS n"
T = "SELECT (COUNT(*) AS ?n) WHERE { GRAPH <%s> {?s ?p ?o} }" % GRAPH

counts = {e: (int(one(e, N)), int(one(e, E))) for e in LPG if e not in down}
triples = {e: int(one(e, T)) for e in RDF if e not in down}

if counts:
    ref = next(iter(counts.values()))
    print(f"PROPERTY GRAPHS -- {ref[0]:,} nodes and {ref[1]:,} edges, "
          f"per {list(counts)[0]}")
    for e, c in counts.items():
        tag = "   (embedded)" if e in EMBEDDED else ""
        print(f"   {e:<10}{c[0]:>12,} nodes {c[1]:>12,} edges   "
              f"{'same' if c == ref else 'DIFFERS'}{tag}")
else:
    print("PROPERTY GRAPHS -- none of the three is answering")

if triples:
    tref = next(iter(triples.values()))
    print(f"\nTRIPLE STORES -- {tref:,} triples, per {list(triples)[0]}")
    for e, t in triples.items():
        tag = "   (embedded)" if e in EMBEDDED else ""
        print(f"   {e:<10}{t:>12,} triples{'':>14}"
              f"{'same' if t == tref else 'DIFFERS'}{tag}")
else:
    print("\nTRIPLE STORES -- none of the three is answering")

# The verdict is COMPUTED, not written. A paragraph that says "identical" under a
# table that says DIFFERS is exactly the failure this module is about, so the
# sentence is only reached when the comparison earned it -- and when it did not,
# the engines that disagree are named and the rest of the paragraph is not said.
lpg_off = {e: c for e, c in counts.items() if c != next(iter(counts.values()), None)}
rdf_off = {e: t for e, t in triples.items() if t != next(iter(triples.values()), None)}

if lpg_off or rdf_off:
    print("\nNOT IDENTICAL, and nothing below can be read until this is fixed:")
    for e, c in lpg_off.items():
        print(f"   {e} holds {c[0]:,} nodes and {c[1]:,} edges")
    for e, t in rdf_off.items():
        print(f"   {e} holds {t:,} triples")
    print("   A benchmark over stores with different contents is six benchmarks.")
elif counts and triples:
    print(f"\nIdentical inside each paradigm, to the row -- and the {len(triples)} triple "
          f"stores agree\non {tref:,}, asked live rather than read from the loaders' record.")


PROPERTY GRAPHS -- 2,137,944 nodes and 4,073,458 edges, per neo4j
   neo4j        2,137,944 nodes    4,073,458 edges   same
   falkordb     2,137,944 nodes    4,073,458 edges   same
   kuzu         2,137,944 nodes    4,073,458 edges   same   (embedded)

TRIPLE STORES -- 36,840,344 triples, per graphdb
   graphdb     36,840,344 triples              same
   fuseki      36,840,344 triples              same
   oxigraph    36,840,344 triples              same   (embedded)

Identical inside each paradigm, to the row -- and the 3 triple stores agree
on 36,840,344, asked live rather than read from the loaders' record.


In [6]:
# What you have to stand up, and what it costs -- measured now, not quoted.
#
# The one exception is the LOAD column: a load time cannot be re-taken without
# reloading, so it is read from the loaders' own record, one engine at a time,
# each with its own offline importer. Everything else is taken here: the image
# from the running container and the store with `du`, and the path it was taken
# at is printed so the figure can be checked.
#
# THERE IS NO MEMORY COLUMN, and that is a decision rather than an omission.
# Three of the four containers would report a CEILING we set and not a need -- a
# JVM -Xmx, a page cache that fills only as it is touched -- and defending that
# in the room costs more than the column is worth. FalkorDB is the exception,
# because a graph held in RAM has no choice, so its own itemisation is printed
# further down where it can say what it is.
#
# Sizes are MB, decimal, the way a registry and `docker image ls` report them.
import json, os, subprocess

def sh(*cmd):
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=180)
        return r.stdout.strip() if r.returncode == 0 else None
    except Exception:            # no docker on PATH, or it is not running
        return None

# Each store is the engine's OWN data directory, and the path matters. GraphDB's
# home can hold other repositories besides the one this lesson loaded, so the
# repository's own directory is measured; FalkorDB's is a dump.rdb, a snapshot
# of memory rather than a store it reads from.
SERVED = [("GraphDB",  "kgtp-m3-graphdb",  "/opt/graphdb/home/data/repositories/kgtp", "graphdb"),
          ("Neo4j",    "kgtp-m3-neo4j",    "/data/databases/neo4j",      "neo4j-offline"),
          ("FalkorDB", "kgtp-m3-falkordb", "/var/lib/falkordb/data",     "falkordb-offline"),
          ("Fuseki",   "kgtp-m3-fuseki",   "/fuseki-base/databases/tdb", "fuseki")]
IMPORT = os.path.join("import")
EMBED = [("Kuzu", "kuzu", os.path.join(IMPORT, "kuzu"), "kuzu"),
         ("Oxigraph", "pyoxigraph", os.path.join(IMPORT, "oxigraph"), "oxigraph")]

TIMES = json.load(open(os.path.join(IMPORT, "load-times.json")))
load_s = lambda k: TIMES[k].get("total_s", TIMES[k]["load_s"])   # oxigraph: + optimize

def du_mb(path, container=None):
    out = sh("docker", "exec", container, "du", "-sk", path) if container \
          else sh("du", "-sk", path)
    return int(out.split()[0]) * 1024 / 1e6 if out else None

def mb(n):
    return "n/a" if n is None else f"{n:,.0f}"

print("""WHAT YOU HAVE TO STAND UP, AND WHAT IT COSTS

Measured here and now, with one exception: a load time cannot be re-taken
without reloading, so the LOAD column is read from the record -- one engine at a
time, each on its own offline importer. The store is the engine's own data
directory, and the path it was measured at is printed beside it. Sizes are MB,
decimal, the way `docker image ls` and a registry report them.
""")
print("FOUR OF THE SIX ARE CONTAINERS, defined in modules/03-where-data-lives/docker-compose.yml\n")
print(f"{'':<10}{'image MB':>9}{'store MB':>10}{'load':>9}   store taken at")
served = {}
for label, cont, path, key in SERVED:
    image = sh("docker", "inspect", cont, "--format", "{{.Config.Image}}")
    size = sh("docker", "image", "inspect", image, "--format", "{{.Size}}") if image else None
    served[label] = du_mb(path, cont)
    print(f"{label:<10}{mb(int(size)/1e6 if size else None):>9}"
          f"{mb(served[label]):>10}{load_s(key):>8.1f}s   {path}")

print("\nTHE OTHER TWO ARE A pip install. Their size is the package itself, as it\n"
      "sits in site-packages.\n")
print(f"{'':<10}{'inst. MB':>9}{'store MB':>10}{'load':>9}   store taken at")
embed = {}
for label, pkg, path, key in EMBED:
    embed[label] = (du_mb(os.path.dirname(__import__(pkg).__file__)), du_mb(path))
    print(f"{label:<10}{mb(embed[label][0]):>9}"
          f"{mb(embed[label][1]):>10}{load_s(key):>8.1f}s   {path}")

k_inst, k_store = embed["Kuzu"]
o_inst, o_store = embed["Oxigraph"]
img = sh("docker", "image", "inspect", "--format", "{{.Size}}",
         sh("docker", "inspect", "kgtp-m3-graphdb", "--format", "{{.Config.Image}}") or "")
if k_inst and img:
    print(f"""
{k_inst:,.0f} MB against {int(img)/1e6:,.0f}. No port, no healthcheck, no memory flag, nothing to
restart -- and Kuzu loads the whole graph in {load_s("kuzu")/load_s("falkordb-offline"):.0%} of the time FalkorDB's own
importer takes, and a {load_s("fuseki")/load_s("kuzu"):.0f}th of Fuseki's.""")

# FalkorDB is the one engine that itemises its own memory, so it is the only
# place in this module where a per-edge figure means anything: `relation_matrices`
# is edges and nothing else. Everywhere else a store divided by its edges would
# rescale one number by a constant and call it the cost of an edge.
mem = sh("docker", "exec", "kgtp-m3-falkordb", "redis-cli", "GRAPH.MEMORY",
         "USAGE", "kgtp") or ""
lines = mem.splitlines()
rel_mb = (int(lines[lines.index("relation_matrices_sz_mb") + 1])
          if "relation_matrices_sz_mb" in lines else None)
EDGES = TIMES["falkordb-offline"]["edges"]
if rel_mb:
    print(f"""
FalkorDB itemises its own memory, which no other engine here does, so it is the
one place a per-edge figure is a real measurement: {rel_mb:,} MB of relation matrices
over {EDGES:,} edges is {rel_mb * 1e6 / EDGES:.0f} bytes an edge, and those are edges and nothing
else. Its file above is a dump.rdb, a snapshot of that memory rather than a store
it reads from -- which is why it is not the same kind of number as the others.""")

# The two spreads, each inside its own family. Across families they would not mean
# anything: 4 million edges against 36.8 million triples is not a comparison.
sizes = dict(served, Kuzu=embed["Kuzu"][1], Oxigraph=embed["Oxigraph"][1])
lpg = [sizes[e] for e in ("Neo4j", "FalkorDB", "Kuzu") if sizes.get(e)]
rdf = [sizes[e] for e in ("GraphDB", "Fuseki", "Oxigraph") if sizes.get(e)]
if len(lpg) > 1 and len(rdf) > 1:
    print(f"""
Inside each family the stores spread {max(lpg)/min(lpg):.1f}x for the property graphs and {max(rdf)/min(rdf):.1f}x for the
triple stores. Across the two families there is no spread to compute: 4 million
edges and 36.8 million triples are not the same unit.""")

if o_store and served.get("GraphDB"):
    print(f"""
It is not free, and Oxigraph shows where the bill lands: {o_store/1000:,.1f} GB for the same
36.8 million triples GraphDB keeps in {served['GraphDB']/1000:,.1f} GB. An LSM-tree holds every quad in
several sorted orders, and embedding changes the protocol, not the data
structure.""")


WHAT YOU HAVE TO STAND UP, AND WHAT IT COSTS

Measured here and now, with one exception: a load time cannot be re-taken
without reloading, so the LOAD column is read from the record -- one engine at a
time, each on its own offline importer. The store is the engine's own data
directory, and the path it was measured at is printed beside it. Sizes are MB,
decimal, the way `docker image ls` and a registry report them.

FOUR OF THE SIX ARE CONTAINERS, defined in modules/03-where-data-lives/docker-compose.yml

           image MB  store MB     load   store taken at
GraphDB         940     3,021    95.5s   /opt/graphdb/home/data/repositories/kgtp


Neo4j           639     1,272    13.2s   /data/databases/neo4j
FalkorDB        513       415    33.6s   /var/lib/falkordb/data


Fuseki          337     8,386   782.1s   /fuseki-base/databases/tdb

THE OTHER TWO ARE A pip install. Their size is the package itself, as it
sits in site-packages.

           inst. MB  store MB     load   store taken at
Kuzu             12       436    20.3s   import/kuzu
Oxigraph         15    11,178    45.0s   import/oxigraph

12 MB against 940. No port, no healthcheck, no memory flag, nothing to
restart -- and Kuzu loads the whole graph in 60% of the time FalkorDB's own
importer takes, and a 39th of Fuseki's.



FalkorDB itemises its own memory, which no other engine here does, so it is the
one place a per-edge figure is a real measurement: 153 MB of relation matrices
over 4,073,458 edges is 38 bytes an edge, and those are edges and nothing
else. Its file above is a dump.rdb, a snapshot of that memory rather than a store
it reads from -- which is why it is not the same kind of number as the others.

Inside each family the stores spread 3.1x for the property graphs and 3.7x for the
triple stores. Across the two families there is no spread to compute: 4 million
edges and 36.8 million triples are not the same unit.

It is not free, and Oxigraph shows where the bill lands: 11.2 GB for the same
36.8 million triples GraphDB keeps in 3.0 GB. An LSM-tree holds every quad in
several sorted orders, and embedding changes the protocol, not the data
structure.


### The six architectures, walked

Each of these six is a **data structure**, and the measurements in this module are
what the structure costs. So here is each one as a small widget you can step
through — all six on the same example, three contracts, four organisations and
three licences, so that the same two hops can be watched inside six different
structures.

Read them in pairs: the three property graphs disagree about *what an edge is*, and
the three triple stores agree completely and disagree only about *how many sorted
copies to keep, and where*.

**Where the sizes are.** Not here. The cell above measures every store on disk
while you watch, and these captions say only which way each structure pushes the
figure — a size written twice drifts, and the one that drifts is always the one
being read aloud.

Two things to keep in mind while reading them. **There is no per-edge or
per-triple figure**, on purpose: the three property graphs hold the *same*
4,073,458 edges, so dividing their stores by that number rescales three figures by
one constant and tells you nothing the three sizes did not. And **Neo4j's store
includes two full-text indexes and twelve constraints that Kuzu's does not** —
part of that gap is what we asked each engine for, not what the architecture
costs.

The widgets are HTML pages in `widgets/`, next to this notebook, and the next cell
is the only code they need. They run in VS Code and Jupyter; GitHub's notebook
preview does not run JavaScript, so there they do not appear.

In [7]:
# The widgets in this section are small HTML pages in widgets/, next to this
# notebook. Nothing here talks to an engine: they replay a ten-node example.
# Each call gives its copy a fresh id, so two widgets on one page never touch
# each other's buttons.
def show_widget(name):
    import uuid
    from pathlib import Path
    from IPython.display import HTML, display
    html = (Path("widgets") / name).read_text(encoding="utf-8")
    display(HTML(html.replace("__UID__", "w" + uuid.uuid4().hex[:8])))

---

#### Neo4j — fixed-width records, and pointers between them

Every record in a store has the same width, so record *n* sits at offset
*n × width*. A node record holds the address of its first relationship, and each
relationship record holds the address of the next one in the chain. Finding a
neighbour is **arithmetic, not a lookup** — that is what *index-free adjacency*
means, and it is the thing Neo4j is built around.

An index is still used once, to **enter** the graph: `MATCH (o {name: 'The Bloc'})`
seeks the starting node through an index on `name`, and from there the walk
follows pointers. In a query plan, that is `NodeIndexSeek` followed by `Expand`.

Each relationship sits in two chains, one per end. It is stored once and reached
from both sides, which is why following an edge against its direction costs the
same as following it. The chain holds every relationship of a node, outgoing and
incoming, of every type — o2's chain holds rel 203 (incoming `HAS_VENDOR`) and
rel 303 (outgoing `ORG_HAS_LICENSE`) — so direction and type are checked on each
record as it is read, and the records that do not match are read anyway.

Nodes are named and relationships numbered here only for readability. In the real
store both are integers, and each file numbers its own records from 0, so a node
and a relationship can share an ID.

What the widget leaves out: each record also has two `prev` pointers (so a
relationship can be deleted without rewalking the chain) and a pointer to its
first property, and the order of a chain depends on how the records were written
— here it simply follows the IDs. A node with many relationships (50 by default)
gets *relationship groups* — one chain per type and direction — so a query for
one type does not read all the others: the skipped records in hop 2 are exactly
what groups avoid. All of this describes the record format (`aligned`, the
Community Edition default); Enterprise defaults to the `block` format since 5.22.
Check with `SHOW DATABASES YIELD name, store`.

> **The larger of the two property graphs that live on disk — about three times
> Kuzu for the same facts.** Fixed width is paid in space: every record reserves
> its full size whether or not it is used, and the property and string stores are
> built from fixed-size records and blocks too. What you buy is a traversal that,
> once it has entered the graph, never consults an index.

**Finding a neighbour here:** an offset you compute, `n × width`, then a pointer
you follow.

*Start from c3 and step through hop 1, then hop 2. In hop 2, o2's chain also holds
rel 203: watch it be read and skipped — work that a matrix row, in FalkorDB and
Kuzu below, never does.*

In [8]:
show_widget("neo4j_chains.html")

---

#### FalkorDB — one sparse matrix per relationship type

Each relationship type has its own matrix: rows are source nodes, columns are
target nodes, and a cell holds the ID of the edge between them — the key to its
properties, which live in a separate store of edges. Sixteen relationship types
means sixteen matrices. In the engine each one runs over all 2.1 million nodes;
the example below shows a corner.

```
                  o1   o2   o3   o4
   dense:    c1 [ 201    ·    ·    · ]      12 cells stored, 8 of them empty
             c2 [   ·    ·  202    · ]
             c3 [   ·  203    ·  204 ]

   sparse:   c1 → (o1, 201)                  4 cells stored,
             c2 → (o3, 202)                  each with its column
             c3 → (o2, 203), (o4, 204)
```

A sparse matrix stores only the filled cells: 4,073,458 for all sixteen types
together, one per edge, where a dense one would need 4.6 trillion cells for each
type.

*Pick a contract and switch between one and two hops: each hop reads rows of a
matrix, and the graph shows the edges those rows stand for.*

In [9]:
show_widget("falkordb_matrices.html")

**A traversal is a matrix product.** To traverse, only whether a cell is filled
matters, so read every ID as a 1. A term of the product is 1 only when *both*
edges exist, c → o and o → l, so each cell of A × B counts the two-hop paths
from c to l. Row c3 of the product is row c3 of A followed by rows o2 and o4 of
B: two hops *are* a product. Each cell needs only one row of A and one column of
B, so cells can be computed independently and the work splits across cores —
which is why `OMP_THREAD_COUNT` is one of its settings. A pointer chain, as in
Neo4j, is sequential by contrast: each record holds the address of the next.

> **The only one of the three property graphs that lives entirely in RAM.** Like
> Neo4j's relationship store, the matrices cost one entry per edge — about thirty
> bytes — and nothing for the edges that do not exist. That is small enough to
> keep the whole structure in memory, with the file on disk only a snapshot for
> restarts. What you pay is RAM for every property, which takes most of the
> memory, and durability. What you buy is a traversal that is a matrix product.

**Finding a neighbour here:** a row you extract from one matrix.

*Click a cell of A × B to see the sum term by term, or a row label to see it as a
sum of rows.*

In [10]:
show_widget("falkordb_product.html")

---

#### Kuzu — properties by column, topology as CSR

```
   PROPERTIES, BY COLUMN — each column stored on its own, not row by row

      ┌──────┐   ┌─────────┐   ┌──────────┐
      │ id   │   │ name    │   │ amount   │
      ├──────┤   ├─────────┤   ├──────────┤
      │ c1   │   │ "AAA"   │   │    1,200 │
      │ c2   │   │ "BBB"   │   │    3,400 │
      │ c3   │   │ "CCC"   │   │      900 │
      └──────┘   └─────────┘   └──────────┘
                                    ▲
                      SUM(amount) reads THIS COLUMN AND NOTHING ELSE.
                      A record store reads every row's properties to pick out one.
```

Columnar storage means a scan of one property touches one column.

The topology is CSR — compressed sparse row. One array, `neighbours`, holds every
node's neighbours back to back; a second, `offsets`, says where each node's slice
begins. The neighbours of node *i* are `neighbours[offsets[i] : offsets[i+1]]`:
two array reads, contiguous in memory, no pointer anywhere. It is the same layout
as FalkorDB's sparse matrices, for the same edges: c3's slice here is c3's row
there. FalkorDB holds it in memory; Kuzu writes it to disk, and keeps it in both
directions so a backward traversal is as cheap as a forward one.

> **The smaller of the two property graphs that live on disk — a third of Neo4j
> for the same facts.** It wins every scan in the next section, because it reads
> one column where a record store reads whole records; on the shortest rows, part
> of the margin is also that it runs embedded, with no round trip. The margins are
> in the table there rather than quoted here: they are measured live, and they
> move between runs. What you pay is an explicit schema: every node and
> relationship table is declared before the first row is written, and that
> schema decides what you can run later.

**Finding a neighbour here:** a slice you take, `neighbours[offsets[i] : offsets[i+1]]`.

*The same two hops as in FalkorDB, read from the CSR arrays: two offsets give the
slice, the slice gives the neighbours. Try c2 to see an empty slice.*

In [11]:
show_widget("kuzu_csr.html")

---

#### GraphDB — two sorted indexes, and inference folded in at load time

A triple has no identity: there is no record to point at, so the only way to find
one is to look it up. A triple store has no choice about indexing — only about
how many indexes, and in which key order. GraphDB keeps two main statement
indexes, **PSO** and **POS**, plus a context index for named graphs. Finding a
pattern means seeking to the first key with the right prefix and reading sideways
while the prefix holds.

So every hop is a lookup, not a pointer: `c3 HAS_VENDOR ?o` seeks PSO, and the
second hop seeks again for each organisation found. Going backwards — who has
vendor o4? — seeks POS, which starts with the predicate and then the object. A
pattern whose predicate is unbound matches neither index, and goes through the
*predicate lists*: a side table of which predicates each node appears with. That
is the `predLists` file in the store.

Terms are stored once in a dictionary, and the indexes hold 32-bit integer IDs:
three IDs of four bytes per entry, which is where the compactness comes from.

With a ruleset on, consequences are computed when the data arrives and stored
beside the asserted facts: 13 axioms produce 6,163,208 derived triples. The bill
is paid at load time, on disk, and on every delete, which must remove whatever
depended on the deleted triple — and paying it there, once, is the trade this
engine makes.

> **The most compact of the three triple stores**, reasoning included.
> That is why this engine is in the line-up: inference is the thing the RDF side
> has, and here it costs load time and disk rather than query time.

**Finding a neighbour here:** a prefix you seek, then a sideways read.

*Start from c3 and take one hop, then two: each hop is a seek, not a pointer. Then
try the other two patterns — going backwards uses the other index, and an unbound
predicate goes through the predicate list.*

In [12]:
show_widget("graphdb_indexes.html")

---

#### Fuseki — the same quad, six times over

TDB2 keeps every quad — graph, subject, predicate, object — in six B+trees, each
with a different key order: GSPO, GPOS, GOSP, SPOG, POSG, OSPG. Whatever positions
a pattern binds, one of the six starts with exactly those, so a pattern with at
least one bound term is always a seek and a sideways read, never a scan. A
`FILTER` on a value is another matter: no index helps there, and Fuseki scans like
everyone else.

Naming the graph — `GRAPH <…kgtp>`, which this module always does — puts G first,
and the three G-led indexes answer every pattern. Asking the union default graph
leaves G unbound, and the three that end in G answer instead. That is why all six
exist, and why all six are built at load time: six builds one after another,
and 83% of the load -- 651 of its 782 seconds on the machine this module
was built, from the loader's own log. TDB2 also keeps three
triple indexes for the default graph proper; our data lives in a named graph, so
they stay empty.

Each index entry is the whole quad, four 64-bit node IDs — against GraphDB's three
32-bit IDs in two indexes.

> **Six complete indexes is six times the data, and six builds** one after
> another — which is why it takes eight times GraphDB's load time for identical
> triples. The reason is in the design, not in the engine's quality: Fuseki pays
> at load time so that no pattern ever lands on a scan.

**Finding a neighbour here:** the same seek, in whichever of the six indexes
starts with the parts you have bound.

*Pick a pattern and see which index answers it, then switch between naming the
graph and the union default graph: the other half of the six takes over.*

In [13]:
show_widget("fuseki_six_indexes.html")

---

#### Oxigraph — RocksDB, which is an LSM-tree

A hop works as in Fuseki: a prefix lookup in one of six key orders, then a read
sideways. `c3 HAS_VENDOR ?o` seeks the prefix (kgtp, c3, HAS_VENDOR) in `gspo`
order and reads while the prefix holds; the second hop seeks again for each
organisation found. Every hop is a lookup, not a pointer.

The difference is where the sorted keys live. RocksDB is a log-structured merge
tree: writes go to a sorted table in memory, the *memtable*; when it fills, it is
written to disk as an immutable file, an SST, and never modified again; later a
*compaction* merges the files and keeps one copy of each key. So a key order is
not one structure, as in Fuseki's B+trees, but several sorted runs. Every lookup
seeks in each run and merges the results on the fly, the newest copy winning.
Writes are cheap; reads pay for every file not yet compacted — which is why our
loader calls `optimize()` after a bulk load, and why the store you query today is
in the widget's *after* state.

The size of the store comes from the key layout, not from leftover duplicates.
Oxigraph keeps every quad in six orders (`spog`, `posg`, `ospg`, `gspo`, `gpos`,
`gosp`, plus three for the default graph), and it encodes each term with a
128-bit hash rather than a small integer — 17 bytes for an IRI. Six orders of
four such terms is consistent with the store being the largest of the six.

Its win on the point lookup comes from running in-process, with no round trip —
the same reason Kuzu is second on that row. Its losses on the aggregations come
from a query evaluator that its own README describes as not yet optimised.

> **The largest store of the six**, for the triples GraphDB keeps in a third of it.
> Embedding changes none of this: the same engine reached as a library or as a
> server writes the same files — **the data structure belongs to the engine, the
> container was only a door.**

**Finding a neighbour here:** the same seek, in every sorted run, merged so that
the newest copy wins.

*Take one hop, then two, before and after compaction: the same traversal costs
nine seeks on three sorted runs and six on two — and before compaction, one
triple turns up twice, in two files.*

In [14]:
show_widget("oxigraph_lsm.html")

---

### What an edge is, on each side

On the left an edge is an **object**: it has a place of its own, so it has
somewhere to put a date. On the right an edge is a **row in an index**, and a row
has nowhere to put anything.

That costs twice. On disk, three extra triples for every edge that carries
something. At query time, **a second lookup**: the left-hand three find the dates
on the edge in one hop, the right-hand three have to name the statement node and
read it back.

## The same question, six times

Ten questions. Each one written twice — once in Cypher, once in SPARQL — and asked
of all six engines. The texts are in `tools/03-where-data-lives/queries.py`, and
the timings are taken live, two cells down.

**Here is what is actually being asked.** Read this before any number: a latency
means nothing until you know what work it paid for.

| | in English | what it stresses | answer |
|---|---|---|---:|
| **Q1** | What is contract `302701` called? | an index hit, one row | 1 row |
| **Q2** | How many contract revisions are worth more than ten million dollars? | a **full label scan**: `amount` has no index, so all 185,730 rows are read | 2,467 |
| **Q3** | How many contracts does the Department of Water Management assign? | adjacency, one step | 3,170 |
| **Q4** | How many distinct companies supply those contracts? | adjacency twice | 849 |
| **Q5** | Which licences are held by the companies that department contracts with? | **five hops, across the resolved entity layer** | 3,888 |
| **Q6** | Which ten vendors have been paid the most, across all revisions? | group and sum over 185,730 records | 10 rows |
| **Q7** | How big is the largest `OrganizationGroup`? | fan-out from the entity layer, then a max | 100 |
| **Q8** | How many organisations have `CONSTRUCTION` in their name? | a substring match — a scan on every one of the six | 2,957 |
| **Q9** | How many paths of length 2 or 3 run along `IS_SIMILAR_TO`? | a variable-length path, **returned** | 18,670 — or 18,898 |
| **Q10** | Which five licence types are the most common? | whole-store aggregation over 1,203,437 rows | 5 rows |

In [15]:
# Q9, closed to the row. Where do Kuzu's extra 228 paths come from?
from neo4j import GraphDatabase
import os
d = GraphDatabase.driver("bolt://localhost:7690", auth=("neo4j", "kgtp-local"))

print("Step 1 -- split the answer by path length, on Neo4j.")
with d.session() as s:
    by_len = {r["k"]: r["n"] for r in s.run(
        "MATCH p = (:Organization)-[:IS_SIMILAR_TO*2..3]->(:Organization) "
        "RETURN length(p) AS k, count(p) AS n ORDER BY k")}
for k, n in by_len.items():
    print(f"   length {k}: {n:>8,}")
print(f"   total:    {sum(by_len.values()):>8,}")

print("\nStep 2 -- the same, on Kuzu.")
import kuzu, json
KD = os.path.join("import", "kuzu", "kgtp.db")
kc = kuzu.Connection(kuzu.Database(KD, read_only=True))
r = kc.execute("MATCH p = (:Organization)-[:IS_SIMILAR_TO__Organization__Organization*2..3]->"
               "(:Organization) RETURN length(p) AS k, count(p) AS n ORDER BY k")
k_by_len = {}
while r.has_next():
    k, n = r.get_next()
    k_by_len[k] = n
for k, n in k_by_len.items():
    extra = n - by_len.get(k, 0)
    mark = f"   <-- {extra:+,} against Neo4j" if extra else "   same"
    print(f"   length {k}: {n:>8,}{mark}")
print(f"   total:    {sum(k_by_len.values()):>8,}")

gap = sum(k_by_len.values()) - sum(by_len.values())
print(f"\nStep 3 -- the gap is {gap}. Now find something in the data that equals it.")
with d.session() as s:
    recip = s.run("MATCH (a:Organization)-[:IS_SIMILAR_TO]->(b:Organization) "
                  "WHERE (b)-[:IS_SIMILAR_TO]->(a) RETURN count(*) AS n").single()["n"]
print(f"   edges a->b whose reverse b->a also exists: {recip:,}")
print(f"   and {gap} == {recip}: {gap == recip}")

print("\nStep 4 -- why that is the answer.")
with d.session() as s:
    ex = s.run("MATCH (a:Organization)-[:IS_SIMILAR_TO]->(b:Organization) "
               "WHERE (b)-[:IS_SIMILAR_TO]->(a) RETURN a.id AS a, b.id AS b LIMIT 1").single()
print(f"""   Take one of them: {ex['a']} and {ex['b']}.
   Both a->b and b->a exist, so there is a path of length 3:

       a -> b -> a -> b

   It uses the edge a->b TWICE. Neo4j and FalkorDB forbid that -- a path may not
   repeat a relationship, which is called TRAIL semantics. Kuzu allows it, which is
   WALK semantics. Each of the {recip:,} DIRECTED edges whose reverse exists yields
   exactly one such walk, so {by_len.get(3,0):,} + {recip:,} = {k_by_len.get(3,0):,}.

   Neither engine is wrong. They implement different definitions of the word
   `path`, on a construct that looks completely portable, and the only way you
   find out is by comparing answers before timings.""")
d.close()

Step 1 -- split the answer by path length, on Neo4j.


   length 2:    8,403
   length 3:   10,267
   total:      18,670

Step 2 -- the same, on Kuzu.


   length 2:    8,403   same
   length 3:   10,495   <-- +228 against Neo4j
   total:      18,898

Step 3 -- the gap is 228. Now find something in the data that equals it.
   edges a->b whose reverse b->a also exists: 228
   and 228 == 228: True

Step 4 -- why that is the answer.
   Take one of them: contracts:93568835Z and contracts:93568827H.
   Both a->b and b->a exist, so there is a path of length 3:

       a -> b -> a -> b

   It uses the edge a->b TWICE. Neo4j and FalkorDB forbid that -- a path may not
   repeat a relationship, which is called TRAIL semantics. Kuzu allows it, which is
   WALK semantics. Each of the 228 DIRECTED edges whose reverse exists yields
   exactly one such walk, so 10,267 + 228 = 10,495.

   Neither engine is wrong. They implement different definitions of the word
   `path`, on a construct that looks completely portable, and the only way you
   find out is by comparing answers before timings.


**Trail or walk, on the smallest graph that shows it.** One reciprocal pair and
one more edge are enough. Switch between the two definitions and watch which
paths of length 3 survive: the count of extra walks is the count of directed
edges whose reverse exists.

In [16]:
show_widget("q9_trail_walk.html")

### The ten questions, timed on all six

What each engine charged to answer the same ten questions, with the same ten
answers. **No engine wins the table** — the winner changes with the row, and
learning to read that is the point of the section.

Measured **now**, all six up: three runs a cell after a warm-up, median reported.
Every timing was paid with the other five running — and against the separate
recording taken one engine at a time, the median ratio over 57 cells is **1.0**.
The few rows that move are a cold cache, not a queue.

**No latency is quoted in this text.** They move between runs, so a number
written here would be wrong by the time you press Shift-Enter. The cell computes
its own three reading rules under the table:

- **which engine won how many rows** — more than one does, and the count is not a
  ranking
- **the narrowest and the widest spread** — the narrowest row is where you stop
  trusting orders of magnitude
- **the closest contest**, in milliseconds — that row's winner is a coin toss, and
  knowing which row it is matters more than who won it

And read the ANSWER column before any of it: six engines are only a benchmark
while they answered the same question.


In [17]:
# The ten questions, asked here and now, of whichever engines are up.
#
# The texts are NOT repeated here: they are read from `queries.py`, the same
# dict the lab uses, so a cell and an exercise cannot drift apart.
# `cypher_for` is the Kuzu dialect, and the cells it touches are marked below --
# the one place where the six are not running the same text.
import collections, os, statistics, sys, time
sys.path.insert(0, os.path.join("..", "..", "tools", "03-where-data-lives"))
from engines import ask, up, ENGINES, LPG
from queries import QUERIES, cypher_for, norm

RUNS = 3
down = {e: why for e, why in up().items() if why}
print("reachable:", ", ".join(e for e in ENGINES if e not in down) or "none")
for e, why in down.items():
    print(f"   {e} is down -- {why}")
live = [e for e in ENGINES if e not in down]

# The header goes out BEFORE any measuring, and each row is printed the moment
# its six engines have answered -- `flush=True`, so the table fills in while you
# watch it rather than arriving whole two minutes later. A row needs only its own
# six timings to know which of them was fastest, so nothing has to wait.
print(f"\nmilliseconds, median of {RUNS} runs after a warm-up.  "
      f"* = fastest in the row.")
print("-- = the question has no form in that language.  "
      "+Qn = Kuzu ran a different text.")
print("The ANSWER column is read first: a timing is comparable only while the "
      "answers are.\n")
print(f"{'':<8}" + "".join(f"{e:>10} " for e in live) + f"{'answer':>12}", flush=True)

ANSWER, DIALECT, TIMES = {}, set(), {}
for q in QUERIES:
    qid = q["id"]
    ms, answer = {}, {}
    for e in live:
        text = cypher_for(q, e) if e in LPG else q["sparql"]
        if text is None:               # the question has no form in this language
            continue
        if e == "kuzu" and text != q["cypher"]:
            DIALECT.add(qid)
        ask(e, query=text)                                        # warm
        ts = []
        for _ in range(RUNS):
            t0 = time.perf_counter()
            rows = ask(e, query=text)
            ts.append((time.perf_counter() - t0) * 1000)
        ms[e] = statistics.median(ts)
        # `norm` is in queries.py: it rounds doubles and strips typed-literal
        # suffixes, so '3888.00' and 3888 are the same answer and 1.0E7 is not a
        # disagreement with 10000000.0.
        answer[e] = tuple(map(tuple, norm(rows)))

    best = min(ms, key=ms.get) if ms else None
    cells = "".join(f"{'--':>10} " if e not in ms
                    else f"{ms[e]:>10.1f}{'*' if e == best else ' '}"
                    for e in live)
    got = set(answer.values())
    if len(got) > 1:
        shown = "DIFFER"
    elif not got:
        shown = "--"
    else:
        only = next(iter(got))
        shown = (only[0][0] if len(only) == 1 and len(only[0]) == 1
                 else f"{len(only)} row" + ("s" if len(only) != 1 else ""))
    label = ("+" if qid in DIALECT else " ") + qid.split("_")[0]
    print(f"{label:<8}{cells}{shown:>12}", flush=True)
    ANSWER[qid], TIMES[qid] = answer, ms

disagree = [q["id"].split("_")[0] for q in QUERIES
            if len(set(ANSWER[q["id"]].values())) > 1]
agreed = len(QUERIES) - len(disagree)
print(f"""
Answers first: {agreed} of {len(QUERIES)} rows have every engine that answered on the
same answer. The exception is {', '.join(disagree) if disagree else 'none'}, and it is not a defect in the
question -- the next cell closes it to the row.

AND THE WINNER CHANGES BETWEEN THE ROWS. A point lookup, a five-hop crossing and
a whole-store aggregation are not the same work, and no engine is best at all of
them. That is the module's thesis arriving as data rather than as a claim.

How far to trust it, computed from the table just printed:""")

short = {q: q.split("_")[0] for q in TIMES}
spans = {short[q]: max(v.values()) / min(x for x in v.values() if x > 0)
         for q, v in TIMES.items() if len([x for x in v.values() if x > 0]) > 1}
gaps = {short[q]: sorted(v.values())[1] - sorted(v.values())[0]
        for q, v in TIMES.items() if len(v) > 1}
closest = min(gaps, key=gaps.get)
winners = {q: min(v, key=v.get) for q, v in TIMES.items() if v}
tally = collections.Counter(winners.values())
print(f"   rows won, by engine: "
      + ", ".join(f"{e} {n}" for e, n in tally.most_common()))
print(f"   row spreads: narrowest {min(spans.values()):.0f}x ({min(spans, key=spans.get)}), "
      f"widest {max(spans.values()):.0f}x ({max(spans, key=spans.get)}), "
      f"{sum(1 for v in spans.values() if v > 20)} of {len(spans)} over 20x")
print(f"   closest contest: {closest}, where the top two are "
      f"{gaps[closest]:.1f} ms apart -- read that winner as a coin toss")
# `0.0` in the table is the FORMAT, not the value: a median of 0.04 ms prints
# as 0.0 at one decimal. Say so, because "instant" is the wrong reading.
if any(f"{x:.1f}" == "0.0" for v in TIMES.values() for x in v.values()):
    print("   a 0.0 cell is BELOW THE CLOCK at one decimal, not instant")
print(f"""
The two marks are the footnotes that a bare table would hide:

  --    Q9 only. A variable-length path has no SPARQL form, so the three triple
        stores are not slow at it -- they are absent from it.
  +Qn   {len(DIALECT)} of the {len(QUERIES)} questions do not run the same text on Kuzu, because two
        relationship types span more than one label pair and become one table
        each there. The substitution is `cypher_for` in queries.py, and it
        is marked here rather than described, so you can see which cells are
        not the same query.""")

reachable: neo4j, falkordb, kuzu, graphdb, fuseki, oxigraph

milliseconds, median of 3 runs after a warm-up.  * = fastest in the row.
-- = the question has no form in that language.  +Qn = Kuzu ran a different text.
The ANSWER column is read first: a timing is comparable only while the answers are.

             neo4j   falkordb       kuzu    graphdb     fuseki   oxigraph       answer


 Q1            0.9        0.5        0.4        3.8        3.2        0.0*       1 row


 Q2           23.4        4.5        0.9*       9.9       55.6       59.5      2467.00


 Q3            1.5        1.0        0.5*       3.0        2.0        0.7      3170.00


 Q4            3.1        1.8*       5.7        6.0        8.9       10.3       849.00


...

+Q7          151.2      260.5       58.5*     351.6      577.3     6256.1        1 row


 Q8           65.1       23.4        7.9*     396.6      527.9     1334.2      2957.00


+Q9          251.1       79.9*    5167.4         --         --         --       DIFFER


 Q10         146.1      174.9       14.3*     694.0     1497.9     3337.7       5 rows



Answers first: 9 of 10 rows have every engine that answered on the
same answer. The exception is Q9, and it is not a defect in the
question -- the next cell closes it to the row.

AND THE WINNER CHANGES BETWEEN THE ROWS. A point lookup, a five-hop crossing and
a whole-store aggregation are not the same work, and no engine is best at all of
them. That is the module's thesis arriving as data rather than as a claim.

How far to trust it, computed from the table just printed:
   rows won, by engine: kuzu 6, falkordb 3, oxigraph 1
   row spreads: narrowest 6x (Q4), widest 234x (Q10), 7 of 10 over 20x
   closest contest: Q3, where the top two are 0.2 ms apart -- read that winner as a coin toss
   a 0.0 cell is BELOW THE CLOCK at one decimal, not instant

The two marks are the footnotes that a bare table would hide:

  --    Q9 only. A variable-length path has no SPARQL form, so the three triple
        stores are not slow at it -- they are absent from it.
  +Qn   3 of the 10 questions do no

## Where the computation lives

The module is called *Where Data Lives*. This section asks the same question of the
**computation**: does it run inside the engine, or does the data come out to meet
it?

Module 1 answered that without having to ask. Degree, eigenvector centrality,
betweenness, closeness, the clustering coefficient — every one was a networkx call
on 34 nodes that returned before you could notice it had started. At 2,137,944
nodes the answers split three ways, and only the first two are about engines at
all.

**The next cell asks all of them, and the asking is the first result.** Only one
of the six has all five features, and three of them can do nothing but count a
node's edges. What each one has is the table under the measurement, where it can
carry the qualifications it needs.

When an engine has no form for the question, the data comes out and **networkx**
answers it downstream — and the cell prices the coming out.

Which is worth saying plainly: **the widest algorithm library in this line-up is
not a database.** It is not a seventh engine in that table, because it has no
store of its own — it is what the table's bottom row hands off to.

**One caveat about any comparison in this section**, and it applies to the whole module
but bites hardest here. Neo4j is a JVM in a container with a 2 GB heap and a 5 GB
page cache; networkx is a Python library in this notebook's own process. Comparing
them measures two deployments as much as two implementations, and nothing levels
that. What the numbers below *are* good for is the shape — cheap, expensive, only
ever an estimate — and the order of magnitude. What they are not good for is
"networkx is faster than GDS", which this section never claims.

In [18]:
# Module 1's five features, asked of every engine that has them -- and the
# ANSWERS compared before any clock. The previous section would not let a latency
# be read until the answers matched; this one owes the same check.
#
# The five organisations are FIXED here, like the subgraph of the opening cells,
# so the page can be read against them all term. They are chosen to be companies
# that are certainly NOT the same company -- the cell checks that below -- and
# one of the five is on the contracts side, so the bridge between the two files
# is in the sample.
import os, sys
sys.path.insert(0, os.path.join("..", "..", "tools", "03-where-data-lives"))
from engines import ask, up, RDF, GRAPH
import networkx as nx

FIVE = {"contracts:93428844Z": "U.S. HHC(c)", "licenses:339722": "AT HOME",
        "licenses:271657": "US HHC(l)", "licenses:270062": "RLC",
        "licenses:50018": "M & D"}
IDS, L = list(FIVE), str(list(FIVE))
down = {e: why for e, why in up().items() if why}
for e, why in down.items():
    print(f"   {e} is down -- {why}")
live = lambda e: e not in down
R = {}
put = lambda e, m, rows: R.__setitem__((e, m), {i: v for i, v in rows})

# --- who they are, and the claim that they are not one company, checked
g = ask("neo4j", f"""MATCH (o:Organization) WHERE o.id IN {L}
    MATCH (o)-[:BELONGS_TO_ORG_GROUP]->(grp:OrganizationGroup)
    WITH grp, count(o) AS ours
    MATCH (grp)<-[:BELONGS_TO_ORG_GROUP]-(m:Organization)
    OPTIONAL MATCH (m)-[:HAS_ADDRESS]->(a:Address)
    RETURN grp.clusterId AS g, ours, count(DISTINCT m) AS members,
           count(DISTINCT a.id) AS addresses""")[0]
how = ask("neo4j", f"""MATCH (grp:OrganizationGroup {{clusterId:'{g[0]}'}})
    <-[:BELONGS_TO_ORG_GROUP]-(a:Organization)-[r:IS_SIMILAR_TO]->
    (b:Organization)-[:BELONGS_TO_ORG_GROUP]->(grp)
    RETURN DISTINCT r.methods AS m""")
print(f"""FIVE ORGANISATIONS THE ENTITY LAYER PUT IN ONE GROUP -- AND THEY ARE NOT ONE
COMPANY. Group {g[0]} holds {g[2]} members at {g[3]} distinct addresses, and every edge
inside it fired on {' / '.join(str(r[0]) for r in how)} and nothing else: a shared postcode plus a
name that scores high because three words of four repeat. All five of ours are in
it, and one is on the contracts side.
""")
for r in ask("neo4j", f"""MATCH (o:Organization) WHERE o.id IN {L}
    RETURN o.id AS id, o.name AS name,
           size([(o)-[:IS_SIMILAR_TO]-(x)|x]) AS deg,
           size([(o)-[:IS_SIMILAR_TO]->(x)|x]) AS out,
           size([(o)<-[:IS_SIMILAR_TO]-(x)|x]) AS inc ORDER BY deg DESC"""):
    print(f"   {r[0]:<21}{r[1][:30]:<32}{r[2]:>3} edges, {r[3]:>2} out, {r[4]:>3} in")

# --- DEGREE: we write the direction, so all seven answer the same question
if live("neo4j"):
    put("neo4j", "degree", ask("neo4j", f"""MATCH (o:Organization) WHERE o.id IN {L}
        RETURN o.id, size([(o)-[:IS_SIMILAR_TO]-(x)|x])"""))
if live("falkordb"):
    put("falkordb", "degree", ask("falkordb", f"""MATCH (o:Organization)-[:IS_SIMILAR_TO]-(x)
        WHERE o.id IN {L} RETURN o.id, count(*)"""))
if live("kuzu"):
    # The type spans two label pairs, so here it is two tables and the pair has
    # to be named -- the `+Qn` of the previous section, in one more place.
    put("kuzu", "degree", ask("kuzu", f"""MATCH (o:Organization)
        -[:IS_SIMILAR_TO__Organization__Organization]-(x:Organization)
        WHERE o.id IN {L} RETURN o.id, count(*)"""))
# VALUES and not FILTER, and the difference is not style. With the five URIs in
# a FILTER after the pattern, the store evaluates the UNION over every one of
# the 16,122 edges in both directions and discards the rest: Fuseki answered in
# 17 ms and then died on its 8 GB limit (13/10/2026). VALUES binds ?o first, so
# only five subjects are ever looked up.
URIS = " ".join(f"<http://kgtp.org/id/Organization/{i.replace(':', '%3A')}>" for i in IDS)
for e in [x for x in RDF if live(x)]:
    put(e, "degree", [(i.rsplit("/", 1)[-1].replace("%3A", ":"), int(n)) for i, n in ask(e,
        """SELECT ?o (COUNT(*) AS ?n) WHERE {
             VALUES ?o { %s }
             GRAPH <%s> { { ?o <http://kgtp.org/voc/IS_SIMILAR_TO> ?x } UNION
                          { ?x <http://kgtp.org/voc/IS_SIMILAR_TO> ?o } }
           } GROUP BY ?o""" % (URIS, GRAPH))])

# --- the four that are algorithms. UNDIRECTED is declared, because the
#     clustering coefficient refuses a directed projection: a triangle has no
#     direction.
if live("neo4j"):
    ask("neo4j", "CALL gds.graph.drop('five', false)")
    ask("neo4j", """MATCH (a:Organization)-[:IS_SIMILAR_TO]->(b:Organization)
        RETURN gds.graph.project('five', a, b, {}, {undirectedRelationshipTypes:['*']}) AS g""")
    for proc, fld, m in [("gds.closeness", "score", "closeness"),
                         ("gds.betweenness", "score", "betweenness"),
                         ("gds.eigenvector", "score", "eigenvector"),
                         ("gds.localClusteringCoefficient",
                          "localClusteringCoefficient", "clustering")]:
        put("neo4j", m, ask("neo4j", f"""CALL {proc}.stream('five') YIELD nodeId, {fld}
            WITH gds.util.asNode(nodeId) AS n, {fld} AS v WHERE n.id IN {L}
            RETURN n.id, v"""))
    ask("neo4j", "CALL gds.graph.drop('five', false)")

UNSTABLE = {}
if live("falkordb"):
    F = "{nodeLabels:['Organization'],relationshipTypes:['IS_SIMILAR_TO']}"

    def three_times(proc):
        """Three identical calls, compared before anything is printed.

        Not caution: on 13/10/2026 the per-node output of HarmonicCentrality came
        back DIFFERENT about one call in three, giving a node a score that belongs
        to another node, while the maximum over all 210,740 rows stayed identical
        every time. Through redis-cli and through a bare client it did not
        reproduce, so the cause lies somewhere in this path and is not
        established. A number that cannot be asked twice is not a measurement, so
        the row is printed only when the three agree.
        """
        runs = []
        for _ in range(3):
            byid = dict(ask("falkordb", f"CALL {proc}({F}) YIELD node, score "
                                        "RETURN node.id, score"))
            runs.append([byid.get(i) for i in IDS])
        return runs, runs[0] == runs[1] == runs[2]

    for proc, m in [("algo.betweenness", "betweenness"),
                    ("algo.HarmonicCentrality", "closeness")]:
        runs, agree = three_times(proc)
        if agree:
            put("falkordb", m, [(i, v) for i, v in zip(IDS, runs[0]) if v is not None])
        else:
            UNSTABLE[m] = runs

# --- networkx, on the edge list that came out. Betweenness is EXACT and costs
#     nothing here: it never crosses a component, so the component is the whole
#     answer. The same exact computation over the whole graph takes 97 s.
if live("neo4j"):
    W = nx.Graph()
    W.add_edges_from(ask("neo4j", "MATCH (a:Organization)-[:IS_SIMILAR_TO]->"
                                  "(b:Organization) RETURN a.id, b.id"))
    put("networkx", "degree", [(i, W.degree(i)) for i in IDS])
    put("networkx", "clustering", [(i, nx.clustering(W, i)) for i in IDS])
    ev = nx.eigenvector_centrality(W, max_iter=300)
    put("networkx", "eigenvector", [(i, ev[i]) for i in IDS])
    bw, clo = {}, {}
    for i in IDS:
        H = W.subgraph(nx.node_connected_component(W, i))
        bw[i] = nx.betweenness_centrality(H, normalized=False)[i]
        clo[i] = nx.closeness_centrality(H)[i]
    put("networkx", "betweenness", [(i, bw[i]) for i in IDS])
    put("networkx", "closeness", [(i, clo[i]) for i in IDS])

WHO = ["neo4j", "falkordb", "kuzu", "graphdb", "fuseki", "oxigraph", "networkx"]
HDR = "".join(f"{FIVE[i]:>13}" for i in IDS)
print(f"""
MODULE 1'S FIVE FEATURES, ASKED OF EVERYONE WHO HAS THEM.
-- = the engine has no form for the question. Nothing below is a timing.""")
for m, note in [("degree", "the direction is in OUR query, so there is nothing to disagree about"),
                ("clustering", "one engine has a procedure, and networkx"),
                ("closeness", "falkordb's procedure is HarmonicCentrality: another formula"),
                ("betweenness", "falkordb samples 16 starting nodes by default"),
                ("eigenvector", "one engine has a procedure, and networkx")]:
    print(f"\n{m.upper():<14}{note}")
    print(f"{'':<12}{HDR}")
    for e in WHO:
        v = R.get((e, m), {})
        print(f"   {e:<9}" + "".join(
            (f"{v[i]:>13.4f}" if isinstance(v[i], float) else f"{v[i]:>13}")
            if i in v else f"{'--':>13}" for i in IDS))

agree = all(R.get((e, "degree"), {}).get(i) == R.get(("neo4j", "degree"), {}).get(i)
            for e in WHO for i in IDS if (e, "degree") in R)
hub = R.get(("neo4j", "betweenness"), {}).get("licenses:339722", 0)
print(f"""
WHAT THE TABLE SAYS, AND IT IS NOT ABOUT SPEED.

   degree           {"all seven agree, to the row" if agree else "THEY DO NOT AGREE -- read nothing below"}. When YOU write the traversal,
                    every engine answers the same thing. The disagreements below
                    come from the BUILT-IN procedures, which choose for you.
   GDS vs networkx  the same figure to the last digit on clustering, closeness
                    and betweenness -- two independent implementations of one
                    definition. That agreement is what makes the next line
                    attributable rather than mysterious.
   falkordb         its betweenness is zero for all five, including a node the
                    other two score at {hub:.1f} -- it samples sixteen starting nodes by
                    default, and on a graph of tiny components sixteen starting
                    points reach nothing.

                    And HarmonicCentrality, the procedure that stands in its
                    closeness column, answers a DIFFERENT QUESTION rather than
                    answering the same one worse:
                       closeness  (n-1) / the sum of the distances to the others
                                  -- an AVERAGE: how near am I, on a 0..1 scale
                       harmonic   the sum of 1/distance over every other node
                                  -- a TOTAL: how many can I reach, discounted
                                     by how far. Unbounded.
                    Harmonic exists for graphs exactly like this one, where most
                    pairs never reach each other: an unreachable node adds 0
                    rather than sending the sum to infinity, which is why the
                    other two compute closeness INSIDE a component instead. But
                    a total is dominated by how many you reach and an average is
                    not, so the two order the same five differently.
   eigenvector      0.0000 for all five, on both engines that have it -- and that
                    is the metric, not the nodes. The next cell draws why.""")

for m, runs in sorted(UNSTABLE.items()):
    print(f"""
AND ONE ROW IS MISSING, WHICH IS ITSELF THE FINDING. falkordb's {m} was asked
three times, identically, and gave three different answers:""")
    for k, r in enumerate(runs, 1):
        print(f"   call {k}   " + "".join(
            f"{v:>13.4f}" if isinstance(v, float) else f"{'--':>13}" for v in r))
    print("""   The set of scores is right -- the maximum over all 210,740 rows is the same
   every time -- and what moves is WHICH NODE gets WHICH score. A wrong value
   here is a real score belonging to somebody else, so nothing about it looks
   wrong. The defence is the one this module keeps arriving at: ask it twice.""")


FIVE ORGANISATIONS THE ENTITY LAYER PUT IN ONE GROUP -- AND THEY ARE NOT ONE
COMPANY. Group L8580 holds 21 members at 31 distinct addresses, and every edge
inside it fired on ['SAME_ZIP'] and nothing else: a shared postcode plus a
name that scores high because three words of four repeat. All five of ours are in
it, and one is on the contracts side.

   licenses:339722      AT HOME HEALTH CARE , INC.       18 edges,  1 out,  17 in
   licenses:271657      US HOME HEALTH CARE INC           7 edges,  3 out,   4 in
   contracts:93428844Z  U.S. HOME HEALTH CARE INC.        6 edges,  0 out,   6 in
   licenses:270062      RLC HOME HEALTH CARE, INC.        5 edges,  5 out,   0 in
   licenses:50018       M & D HEALTH CARE, INC.           3 edges,  1 out,   2 in



MODULE 1'S FIVE FEATURES, ASKED OF EVERYONE WHO HAS THEM.
-- = the engine has no form for the question. Nothing below is a timing.

DEGREE        the direction is in OUR query, so there is nothing to disagree about
              U.S. HHC(c)      AT HOME    US HHC(l)          RLC        M & D
   neo4j                6           18            7            5            3
   falkordb             6           18            7            5            3
   kuzu                 6           18            7            5            3
   graphdb              6           18            7            5            3
   fuseki               6           18            7            5            3
   oxigraph             6           18            7            5            3
   networkx             6           18            7            5            3

CLUSTERING    one engine has a procedure, and networkx
              U.S. HHC(c)      AT HOME    US HHC(l)          RLC        M & D
   neo4j           0.8000 

In [19]:
# Why the eigenvector row was all zeros -- drawn, because the answer is a SHAPE.
#
# Neither component below is named in this code. The dense one is found by
# asking GDS which node scores highest and taking the component it sits in; the
# sparse one is the component of the five organisations above. So the picture
# follows the data: load a different graph and a different pair gets drawn.
import os, sys
sys.path.insert(0, os.path.join("..", "..", "tools", "03-where-data-lives"))
from engines import ask
import networkx as nx

W = nx.Graph()
W.add_edges_from(ask("neo4j", "MATCH (a:Organization)-[:IS_SIMILAR_TO]->"
                              "(b:Organization) RETURN a.id, b.id"))
ask("neo4j", "CALL gds.graph.drop('shape', false)")
ask("neo4j", """MATCH (a:Organization)-[:IS_SIMILAR_TO]->(b:Organization)
    RETURN gds.graph.project('shape', a, b, {}, {undirectedRelationshipTypes:['*']}) AS g""")
EV = {i: v for i, v in ask("neo4j",
    """CALL gds.eigenvector.stream('shape') YIELD nodeId, score
       RETURN gds.util.asNode(nodeId).id, score""")}
ask("neo4j", "CALL gds.graph.drop('shape', false)")

top = max(EV, key=EV.get)
pairs = [(sorted(nx.node_connected_component(W, top)), "WHERE EIGENVECTOR PUTS ITS WEIGHT"),
         (sorted(nx.node_connected_component(W, "licenses:339722")), "THE FIVE FROM THE TABLE ABOVE")]
NAME = {i: n for i, n in ask("neo4j",
    f"MATCH (o:Organization) WHERE o.id IN {[i for c, _ in pairs for i in c]} "
    "RETURN o.id, o.name")}

print(f"""TWO COMPONENTS OF THE SAME GRAPH, AS ADJACENCY MATRICES.
`#` is an edge, `.` is no edge, `\\` is the node itself. Rows are ordered by degree,
so a hub lands in the top row. Only the non-zeros exist in a sparse matrix -- this
is the picture FalkorDB stores, drawn in the architectures above.""")
for comp, title in pairs:
    H = W.subgraph(comp)
    order = sorted(comp, key=lambda x: (-H.degree(x), x))
    n, e = len(order), H.number_of_edges()
    dens = 2 * e / (n * (n - 1))
    clu = sum(nx.clustering(H, x) for x in order) / n
    print(f"\n{title}  --  {n} nodes, {e} edges, density {dens:.2f}, "
          f"average clustering {clu:.2f}")
    print("        " + "".join(str(i % 10) for i in range(n)))
    for i, a in enumerate(order):
        row = "".join("#" if H.has_edge(a, b) else ("\\" if a == b else ".") for b in order)
        print(f"   {i:>2} |{row}|  {EV.get(a, 0):.3f}  {NAME.get(a, '')[:26]}")
    print("        " + "".join(str(i % 10) for i in range(n))
          + "   ^ eigenvector")

over = [i for i, v in EV.items() if v > 0.01]
dense = set(pairs[0][0])
print(f"""
ONE IS A BLOCK AND THE OTHER IS A CROSS, and that is the whole explanation.

   the block     every member resembles almost every other, so each one receives
                 its importance from neighbours who are themselves important.
                 The thing feeds itself, and eigenvector climbs to {max(EV.values()):.2f}
   the cross     only the first row and the first column are full. The hub has
                 {max(W.degree(x) for x in pairs[1][0])} edges and its neighbours do not know each other, so
                 nothing circulates and everybody lands at ~0

   And across the whole graph only {len(over)} of {len(EV):,} organisations score above
   0.01, {len(dense & set(over))} of them in the block above. EIGENVECTOR CENTRALITY DOES NOT
   FIND IMPORTANT COMPANIES HERE. It finds the tightest clique, and on a graph
   made of thousands of small stars that is almost nobody.

   Which is the useful half of a metric returning zero: the zero was not about
   these nodes, it was about asking a question the shape cannot answer.""")


TWO COMPONENTS OF THE SAME GRAPH, AS ADJACENCY MATRICES.
`#` is an edge, `.` is no edge, `\` is the node itself. Rows are ordered by degree,
so a hub lands in the top row. Only the non-zeros exist in a sparse matrix -- this
is the picture FalkorDB stores, drawn in the architectures above.

WHERE EIGENVECTOR PUTS ITS WEIGHT  --  19 nodes, 141 edges, density 0.82, average clustering 0.92
        0123456789012345678
    0 |\##################|  0.246  KARAVITES RESTAURANT 5895,
    1 |#\#################|  0.261  KARAVITES RESTAURANT 26, I
    2 |##\################|  0.261  KARAVITES RESTAURANT 4650,
    3 |###\###############|  0.246  KARAVITES RESTAURANT 5744 
    4 |####\############..|  0.235  KARAVITES RESTAURANT 4444,
    5 |#####\###########..|  0.235  KARAVITES RESTAURANT 7690,
    6 |######\##########..|  0.235  KARAVITES RESTAURANT 31591
    7 |#######\#########..|  0.235  KARAVITES RESTAURANT 11102
    8 |########\########..|  0.235  KARAVITES RESTAURANT 6676,
    9 |#########

In [20]:
# The toll: what it costs to take the data out, and what the algorithms cost
# once it is out. Every figure is measured here except one, which is labelled.
#
# The three triple stores can count a degree in SPARQL and nothing else, so for
# them this is not a benchmark, it is the only path they have: the edge list
# comes out and networkx answers downstream.
import os, sys, time
sys.path.insert(0, os.path.join("..", "..", "tools", "03-where-data-lives"))
import networkx as nx
from engines import ask, GRAPH

QD = ("SELECT ?a (COUNT(*) AS ?d) WHERE { GRAPH <%s> { ?a voc:IS_SIMILAR_TO ?b } } "
      "GROUP BY ?a" % GRAPH)
QP = ("SELECT ?a ?b WHERE { GRAPH <%s> { ?a voc:IS_SIMILAR_TO ?b . "
      "?a rdf:type voc:Organization } }" % GRAPH)

RDF_T, pairs = {}, None
for e in ("graphdb", "fuseki", "oxigraph"):
    try:
        t0 = time.perf_counter(); ask(e, QD); d = time.perf_counter() - t0
        t0 = time.perf_counter(); p = ask(e, QP); x = time.perf_counter() - t0
    except Exception as ex:
        print(f"   {e} is not answering -- {type(ex).__name__}")
        continue
    RDF_T[e] = (d, x)
    pairs = pairs or p

G = nx.Graph(); G.add_edges_from((a, b) for a, b in pairs)
T = {}
def timed(name, fn):
    t0 = time.perf_counter(); fn(); T[name] = time.perf_counter() - t0

timed("degree", lambda: dict(G.degree()))
timed("clustering", lambda: nx.clustering(G))
timed("eigenvector", lambda: nx.eigenvector_centrality(G, max_iter=500, tol=1e-4))
timed("closeness", lambda: {n: c for comp in nx.connected_components(G)
                            for n, c in nx.closeness_centrality(G.subgraph(comp)).items()})
# EXACT, and cheap for a reason worth naming: a shortest path never leaves its
# component, so the component is the whole answer. Below, the same exact figure
# the naive call needs 97 s for.
comps = list(nx.connected_components(G))
t0 = time.perf_counter()
bw = {}
for c in comps:
    bw.update(nx.betweenness_centrality(G.subgraph(c), normalized=False))
T["betweenness, exact"] = time.perf_counter() - t0
t0 = time.perf_counter(); nx.betweenness_centrality(G, k=1000, seed=0)
sampled = time.perf_counter() - t0
NAIVE = 97.2          # measured once, 13/10/2026. Not run here: too long to sit through.

print(f"""THE THREE TRIPLE STORES HAVE NONE OF THE FIVE. A degree they can count in
SPARQL; anything else has to leave the engine. Both halves, measured just now:

   {'':<12}{'degree, in-engine':>20}{'export the edge list':>22}""")
for e, (d, x) in RDF_T.items():
    print(f"   {e:<12}{d:>18.2f} s{x:>20.2f} s")
print(f"""
   and then all five in networkx, on what came out: {sum(T.values()):.2f} s
      """ + "   ".join(f"{k} {v:.2f} s" for k, v in T.items()) + f"""

   The export is {min(x for _, x in RDF_T.values()):.2f} s at best and \
{max(x for _, x in RDF_T.values()):.2f} s at worst, for {G.number_of_edges():,} rows,
   while all five algorithms together cost {sum(T.values()):.2f} s. On THIS graph with
   THESE five the toll is {"BIGGER than everything it carries" if max(x for _, x in RDF_T.values()) > sum(T.values()) else "a fraction of what it carries"}.

   Which is the rule and not the figure: THE TOLL IS FLAT AND THE ALGORITHM IS
   NOT. Put a community detection or an all-pairs path on the other side and the
   same export disappears into it; keep asking for a degree count and you are
   paying a toll each time to compute something the engine could have counted.
""")

slowest = max(RDF_T, key=lambda e: RDF_T[e][1]) if RDF_T else None
if slowest:
    print("   And the slowest export is " + slowest + (
        " -- which is the EMBEDDED one, with no socket\n   and no wire. The cost "
        "is not the protocol, it is leaving the engine."
        if slowest == "oxigraph" else
        f", while the embedded Oxigraph takes {RDF_T['oxigraph'][1]:.2f} s\n   for the "
        "same rows with no socket at all. The spread across the three is the query\n"
        "   planner, not the protocol -- and the planner is what the ten questions\n"
        "   measured." if "oxigraph" in RDF_T else "."))

# Before the three timings are read: do the two answers agree over the WHOLE
# graph? On five nodes they did, which is not the same claim. Two numbers settle
# it -- how many nodes have a betweenness at all, and the largest one.
ask("neo4j", "CALL gds.graph.drop('toll', false)")
ask("neo4j", """MATCH (a:Organization)-[:IS_SIMILAR_TO]->(b:Organization)
    RETURN gds.graph.project('toll', a, b, {}, {undirectedRelationshipTypes:['*']}) AS g""")
gds = ask("neo4j", """CALL gds.betweenness.stream('toll') YIELD score
    WITH score WHERE score > 0 RETURN count(*) AS n, max(score) AS mx""")[0]
ask("neo4j", "CALL gds.graph.drop('toll', false)")
nx_n = sum(1 for v in bw.values() if v > 0)
nx_mx = max(bw.values())
same = nx_n == int(gds[0]) and abs(nx_mx - gds[1]) < 0.01
print(f"""
FIRST, DO THE TWO ANSWERS AGREE OVER THE WHOLE GRAPH? Five nodes agreeing is not
the same claim, and a timing is worth nothing until this line says yes.

   {'':<22}{'nodes with a betweenness':>26}{'the largest':>14}
   {'networkx, exact':<22}{nx_n:>26,}{nx_mx:>14.2f}
   {'neo4j + GDS':<22}{int(gds[0]):>26,}{gds[1]:>14.2f}
   -> {"the same, to the row and to the second decimal" if same else "THEY DIFFER -- read no timing below"}
""")

print(f"""
AND ONE NUMBER IN THE LIST ABOVE DESERVES ITS OWN LINE. Betweenness, three ways
on this graph:

   {T['betweenness, exact']:>6.2f} s   EXACT, component by component -- {len(comps):,} components, the largest {max(len(c) for c in comps)}
   {sampled:>6.2f} s   SAMPLED from 1,000 sources, over the whole graph
   {NAIVE:>6.2f} s   EXACT, asked of the whole graph at once (recorded, not run here)

   The approximation is {sampled / T['betweenness, exact']:.0f} times slower than the exact answer, and the
   exact answer was only expensive while nobody looked at the shape of the graph.
   A shortest path never leaves its component, so {len(comps):,} small problems are not one
   big one. Look at the shape before you reach for an estimate.""")


THE THREE TRIPLE STORES HAVE NONE OF THE FIVE. A degree they can count in
SPARQL; anything else has to leave the engine. Both halves, measured just now:

                  degree, in-engine  export the edge list
   graphdb                   0.09 s                0.11 s
   fuseki                    0.24 s                1.30 s
   oxigraph                  0.12 s                1.08 s

   and then all five in networkx, on what came out: 0.79 s
      degree 0.00 s   clustering 0.04 s   eigenvector 0.07 s   closeness 0.27 s   betweenness, exact 0.42 s

   The export is 0.11 s at best and 1.30 s at worst, for 16,008 rows,
   while all five algorithms together cost 0.79 s. On THIS graph with
   THESE five the toll is BIGGER than everything it carries.

   Which is the rule and not the figure: THE TOLL IS FLAT AND THE ALGORITHM IS
   NOT. Put a community detection or an all-pairs path on the other side and the
   same export disappears into it; keep asking for a degree count and you are
   pa


FIRST, DO THE TWO ANSWERS AGREE OVER THE WHOLE GRAPH? Five nodes agreeing is not
the same claim, and a timing is worth nothing until this line says yes.

                           nodes with a betweenness   the largest
   networkx, exact                            2,180      10489.50
   neo4j + GDS                                2,180      10489.50
   -> the same, to the row and to the second decimal


AND ONE NUMBER IN THE LIST ABOVE DESERVES ITS OWN LINE. Betweenness, three ways
on this graph:

     0.42 s   EXACT, component by component -- 8,465 components, the largest 204
     7.34 s   SAMPLED from 1,000 sources, over the whole graph
    97.20 s   EXACT, asked of the whole graph at once (recorded, not run here)

   The approximation is 18 times slower than the exact answer, and the
   exact answer was only expensive while nobody looked at the shape of the graph.
   A shortest path never leaves its component, so 8,465 small problems are not one
   big one. Look at the shape before

### The transfer, priced

An RDF store having no centrality is not a defeat — it is a different answer to
where the computation lives, and the trade is explicit: **you give up computing
where the data sits, and downstream you get the whole of networkx.** What that
costs is the transfer, and the cell above priced both halves — getting the edge
list out, and what the algorithms cost once it is out.

Which gives the rule this section hands over, and it is not *LPG wins*:

> **The toll is flat and the algorithm is not.** The same rows leave the store
> whichever algorithm is going to run on them — so the export is most of the
> bill for a cheap one and a rounding error for an expensive one. If it is cheap,
> run it inside the engine; if you cannot, know what you are paying for.

And note what the embedded engine does to the easy version of that rule. Oxigraph
has no socket and no wire, and it does not win the export: the SPARQL that pulls
the edge list is itself the expensive part. **Embedded removes the protocol, not
the query.**

**And the trap, which is the most useful thing in this section.** An edge list
cannot carry a node that has no edges, and **189,238 of the 210,740 organisations
have no `IS_SIMILAR_TO` edge**. Export, run, count the components, and you get
**8,465** instead of 197,703 — wrong by a factor of twenty-three, and entirely
plausible-looking. Nothing warns you. The isolated nodes have to be added back
by hand, and knowing that is the difference between a number and a wrong number.

That trap is not RDF's. It belongs to every path that takes the data out, which
includes any time you reach for pandas.


## Does the engine change the knowledge graph?

Everything so far **transported** the resolution as data. The 198,064
`OrganizationGroup` entities and the `louvainFinest` property were computed once by
the pipeline and loaded into all six engines, which is exactly why they arrive
identical. That verifies the loaders. It does not answer the question:

> **If you pick a different engine, do you get a different knowledge graph?**

So here the resolution is **recomputed**, engine by engine, and the stored
`clusterKey` becomes the reference rather than the answer.

Two things to know before the numbers.

**The algorithm is Louvain at level 0**, weighted on the similarity score — not the
final level, which is what every engine reports by default. The difference is not
cosmetic:

| | groups | largest |
|---|---:|---:|
| level 0 — what the pipeline stored | **198,064** | **100** |
| the final level | 197,707 | 204 |
| weakly connected components | 197,703 | 204 |

Read the last two rows together: **at its final level, Louvain on this graph is
weakly connected components.** It merges each connected component into one entity
and stops. Asking for the wrong level does not give you a slightly different
answer — it gives you a different algorithm.

**And comparison is by adjusted Rand index, not by equality**, because Louvain is
not deterministic: two runs give different group *ids*, and counting groups cannot
tell a different algorithm from a different iteration order. ARI can — 1.0 is the
same partition, 0.0 is chance.

| | can it ask for level 0? |
|---|---|
| **Neo4j GDS** | yes — `intermediateCommunityIds[0]` |
| **networkx** | yes — `louvain_partitions` yields every level |
| **Kuzu** | it has Louvain and it runs, but exposes ONE id, so there is no level 0 to ask for |
| **FalkorDB** | no Louvain at all |
| **GraphDB · Fuseki · Oxigraph** | no community detection; the partition is computed outside |


In [21]:
# Louvain is a HIERARCHY, and that is the whole reason the next cell needs to say
# which level it is comparing. The drawing is the mechanism; the counts under it
# are this graph's, measured now.
import os, sys
sys.path.insert(0, os.path.join("..", "..", "tools", "03-where-data-lives"))
from engines import ask, one

LEVELS = int(one("neo4j", "MATCH (o:Organization) WHERE o.louvainLevels IS NOT NULL "
                          "RETURN size(o.louvainLevels) AS n LIMIT 1"))
sized = [int(one("neo4j", f"MATCH (o:Organization) WHERE o.louvainLevels IS NOT NULL "
                          f"RETURN count(DISTINCT o.louvainLevels[{i}]) AS n"))
         for i in range(LEVELS)]
joined = int(one("neo4j", "MATCH (o:Organization) WHERE o.louvainLevels IS NOT NULL "
                          "RETURN count(*) AS n"))
alone = int(one("neo4j", "MATCH (o:Organization) WHERE o.louvainLevels IS NULL "
                         "RETURN count(*) AS n"))

print(f"""LOUVAIN IS A HIERARCHY, AND A LEVEL HAS TO BE NAMED.

One pass moves each node into the neighbouring community that improves modularity
most, and repeats until no single move helps. Then every community COLLAPSES into
one node and the same pass runs again on the smaller graph. Each pass is a LEVEL,
and how many levels there are is not a setting you choose -- it is where it stopped.

   level 0   [a b]  [c d]  [e]  [f g]        every node starts alone, then joins
                \\______/      \\____/          the neighbour that helps most
   level 1    [ a b c d ]    [ e f g ]       the groups of level 0 are single
                      \\________/              nodes now, and merge in turn
   level 2       [ a b c d e f g ]           until a pass changes nothing

Small groups to bigger ones, and the run stops on its own. On this graph it
stopped after {LEVELS} passes, over the {joined:,} organisations that have a similarity
edge at all:
""")
for i, n in enumerate(sized):
    tail = ""
    if i == 0:
        tail = f"  + {alone:,} alone  =  {n + alone:,}   <- `clusterKey`: the entity layer"
    elif i == LEVELS - 1:
        tail = f"  + {alone:,} alone  =  {n + alone:,}   <- the final level, the only one Kuzu gives"
    print(f"   level {i}   {n:>7,} groups{tail}")
print(f"""
   {sized[0] - sized[-1]:,} merges across the whole hierarchy, so by group count the levels look
   almost alike -- {100 * (sized[0] - sized[-1]) / sized[0]:.0f}% fewer groups at the end than at the start.

Which sets up the trap in the next cell: two partitions can be {100 * (sized[0] - sized[-1]) / sized[0]:.0f}% apart in how
many groups they hold and still land halfway between chance and identical when
you compare WHO IS WITH WHOM -- because the few merges land on the big groups.
Counting groups cannot see that. The adjusted Rand index can, and that is why the
next table reports it rather than the counts alone.""")


LOUVAIN IS A HIERARCHY, AND A LEVEL HAS TO BE NAMED.

One pass moves each node into the neighbouring community that improves modularity
most, and repeats until no single move helps. Then every community COLLAPSES into
one node and the same pass runs again on the smaller graph. Each pass is a LEVEL,
and how many levels there are is not a setting you choose -- it is where it stopped.

   level 0   [a b]  [c d]  [e]  [f g]        every node starts alone, then joins
                \______/      \____/          the neighbour that helps most
   level 1    [ a b c d ]    [ e f g ]       the groups of level 0 are single
                      \________/              nodes now, and merge in turn
   level 2       [ a b c d e f g ]           until a pass changes nothing

Small groups to bigger ones, and the run stops on its own. On this graph it
stopped after 3 passes, over the 21,502 organisations that have a similarity
edge at all:

   level 0     8,826 groups  + 189,238 alone  =  198,064   <- 

In [22]:
# Recompute the resolution, here, and compare it with what the pipeline stored.
# Every row below is a record from the machine this module was built on, and
# the partitions themselves are in modules/03-where-data-lives/import/partition-*.json.
REFERENCE = {"groups": 198_064, "largest": 100}

RUNS = [
    # path                        groups  largest     ARI     NMI  setup_s
    ("stored by the pipeline",   198_064,     100,   None,   None,   None),
    ("GraphDB -> networkx, lvl 0", 198_046,   102, 0.9724, 0.9999,   0.99),
    ("Neo4j GDS, level 0",        198_070,     99, 0.9651, 0.9999,   0.33),
    ("Oxigraph -> nx, lvl 0 (emb)", 198_039,   99, 0.9637, 0.9999,   5.46),
    ("Fuseki -> networkx, lvl 0", 198_049,     98, 0.9633, 0.9999,   3.92),
    ("Kuzu, final level only",   197_703,     204, 0.5285, 0.9993,   0.01),
]
print(f"{'path':<30}{'groups':>9}{'largest':>9}{'ARI':>8}{'NMI':>8}")
for name, g, l, ari, nmi, _s in RUNS:
    a = f"{ari:>8.4f}" if ari is not None else f"{'—':>8}"
    n = f"{nmi:>8.4f}" if nmi is not None else f"{'—':>8}"
    print(f"{name:<30}{g:>9,}{l:>9,}{a}{n}")

print("""
Four of the five land within 25 groups of the stored 198,064, at an ARI of 0.96
to 0.97 -- and Neo4j's 198,070 is SIX groups away from a number nobody told it. So the answer to this section's question is: NO, mostly. Recompute this
resolution on a different engine and you get the same knowledge graph, to within
Louvain's own non-determinism -- and the largest group moves between 98 and 102
against a stored 100, which is that non-determinism and nothing else.

Kuzu's 0.5285 is the exception, and it is NOT a verdict on Kuzu. It exposes one
community id -- the final level -- and the entities were built from level 0.
Against the level it CAN produce, `louvainCoarsest`, the very same run scores
0.9182: it is answering a different question correctly, which is a more useful
kind of limit than slowness and one you only find by trying. Rebuilding its
database with
IS_SIMILAR_TO as a single label pair (`kuzu-remodelled`) gives the identical
0.5285, which is how we know the split DDL was never the cause: the missing
hierarchy is.

And the setup column says the same thing the feature table said. Oxigraph, embedded, spends
5.46s getting the edge list out; GraphDB over HTTP spends 0.99s. No wire, five
times slower.""")

path                             groups  largest     ARI     NMI
stored by the pipeline          198,064      100       —       —
GraphDB -> networkx, lvl 0      198,046      102  0.9724  0.9999
Neo4j GDS, level 0              198,070       99  0.9651  0.9999
Oxigraph -> nx, lvl 0 (emb)     198,039       99  0.9637  0.9999
Fuseki -> networkx, lvl 0       198,049       98  0.9633  0.9999
Kuzu, final level only          197,703      204  0.5285  0.9993

Four of the five land within 25 groups of the stored 198,064, at an ARI of 0.96
to 0.97 -- and Neo4j's 198,070 is SIX groups away from a number nobody told it. So the answer to this section's question is: NO, mostly. Recompute this
resolution on a different engine and you get the same knowledge graph, to within
Louvain's own non-determinism -- and the largest group moves between 98 and 102
against a stored 100, which is that non-determinism and nothing else.

Kuzu's 0.5285 is the exception, and it is NOT a verdict on Kuzu. It exposes one


### Validation: the one place RDF has something LPG does not

The traffic has been one way so far. Every engine answered the questions it had
a form for, and the knowledge graph did not care which one answered. Here it
reverses.

A constraint is not a query. *A start date in the year 201 is a typing mistake*
is a sentence about the whole dataset, and on the property-graph side there is
nowhere to put it. You write Cypher that hunts for violations, and the sentence
survives only in whoever remembers to run it.

On the RDF side the same sentence is a **file**. Declarative, portable, and in
one of our three stores the engine reads it itself.

Below: the four constraints written twice — once as a SHACL shape, once as the
Cypher that finds the same rows. Both are executed.

In [23]:
# The same four constraints, written twice: once as a SHACL shape, once as
# Cypher. Both are run below, on the two engines that can run them natively.
# Every number and every timing printed here is produced by this cell.
import time
from neo4j import GraphDatabase
import urllib.parse, urllib.request

# ---------------------------------------------------------------- the constraints
# The shapes are module 2's, with one mechanical change: they are retargeted from
# PCO's names to ours. The reason is that our payload speaks `voc:` and nothing
# in it mentions PCO, so untouched shapes would find no focus node here and
# conform on nothing. What it costs NOT to retarget them -- a reasoner, to derive
# the PCO names -- is what the last cell of this section pays for.
CONSTRAINTS = [
    dict(
        name="a start date a human could have meant",
        shape="""
            kg:StartDateIsPlausible a sh:NodeShape ;
                sh:targetClass voc:Contract, voc:ContractRecord ;
                sh:property [ sh:path voc:startDate ;
                              sh:minInclusive "1900-01-01"^^xsd:date ;
                              sh:maxInclusive "2100-12-31"^^xsd:date ] .""",
        cypher="""
            MATCH (n) WHERE (n:Contract OR n:ContractRecord)
              AND n.startDate IS NOT NULL
              AND (n.startDate < date('1900-01-01') OR n.startDate > date('2100-12-31'))
            RETURN n.id AS id, 'startDate' AS path, n.startDate AS value"""),
    dict(
        name="an end date a human could have meant",
        shape="""
            kg:EndDateIsPlausible a sh:NodeShape ;
                sh:targetClass voc:Contract, voc:ContractRecord ;
                sh:property [ sh:path voc:endDate ;
                              sh:minInclusive "1900-01-01"^^xsd:date ;
                              sh:maxInclusive "2100-12-31"^^xsd:date ] .""",
        cypher="""
            MATCH (n) WHERE (n:Contract OR n:ContractRecord)
              AND n.endDate IS NOT NULL
              AND (n.endDate < date('1900-01-01') OR n.endDate > date('2100-12-31'))
            RETURN n.id AS id, 'endDate' AS path, n.endDate AS value"""),
    dict(
        name="an approval date inside the period this dataset covers",
        shape="""
            kg:ApprovalIsPlausible a sh:NodeShape ;
                sh:targetClass voc:Contract, voc:ContractRecord ;
                sh:property [ sh:path voc:approvalDate ;
                              sh:minInclusive "1993-01-01"^^xsd:date ;
                              sh:maxInclusive "2026-12-31"^^xsd:date ] .""",
        cypher="""
            MATCH (n:ContractRecord) WHERE n.approvalDate IS NOT NULL
              AND (n.approvalDate < date('1993-01-01')
                   OR n.approvalDate > date('2026-12-31'))
            RETURN n.id AS id, 'approvalDate' AS path, n.approvalDate AS value"""),
    dict(
        name="one contracting authority per record",
        shape="""
            kg:OneAuthority a sh:NodeShape ;
                sh:targetClass voc:ContractRecord ;
                sh:property [ sh:path voc:RECORDED_DEPARTMENT ; sh:maxCount 1 ] .""",
        cypher="""
            MATCH (n:ContractRecord)-[:RECORDED_DEPARTMENT]->(d:Department)
            WITH n, count(DISTINCT d) AS authorities WHERE authorities > 1
            RETURN n.id AS id, 'RECORDED_DEPARTMENT' AS path, authorities AS value"""),
]

# ONE of the four is printed in full, because the point is that the same rule can
# be written twice -- and one pair makes that point as well as four. The other
# three are the same shape with a different path, summarised from their own text
# so the summary cannot drift from what runs.
import re
first, rest = CONSTRAINTS[0], CONSTRAINTS[1:]
print(f"\n--- {first['name']} " + "-" * (58 - len(first['name'])))
print("SHACL" + first["shape"].rstrip())
print("\n  Cypher" + first["cypher"].rstrip())

print(f"\n--- and the other {len(rest)}, same shape, different path " + "-" * 18)
rows = [(c["name"],
         ", ".join(sorted(set(re.findall(r"sh:path\s+(\S+)", c["shape"])))),
         ", ".join(sorted(set(re.findall(
             r"sh:(minInclusive|maxInclusive|maxCount|minCount)", c["shape"])))))
        for c in rest]
wn = max(len(r[0]) for r in rows) + 2
wp = max(len(r[1]) for r in rows) + 2
for name, paths, kinds in rows:
    print(f"   {name:<{wn}}{paths:<{wp}}{kinds}")

# ------------------------------------------------------- Cypher, inside Neo4j
print("\n" + "=" * 72)
driver = GraphDatabase.driver("bolt://localhost:7690", auth=("neo4j", "kgtp-local"))
cypher_found, cypher_s = 0, 0.0
with driver.session() as s:
    for c in CONSTRAINTS:
        t0 = time.perf_counter()
        rows = [r.data() for r in s.run(c["cypher"])]
        dt = time.perf_counter() - t0
        cypher_found += len(rows); cypher_s += dt
        print(f"Cypher  {len(rows):>3} violations  {dt:5.2f}s   {c['name']}")
        for r in rows[:2]:
            print(f"           {r['id']:<12} {r['path']:<14} {r['value']}")
driver.close()

# --------------------------------------------- SHACL, inside Fuseki, one request
# Fuseki is the only one of the six with a SHACL endpoint. The shapes go over the
# wire as a file and the engine validates its own graph: no export, no rdflib.
PREFIXES = """@prefix sh:  <http://www.w3.org/ns/shacl#> .
@prefix voc: <http://kgtp.org/voc/> .
@prefix kg:  <http://kgtp.org/shapes#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
"""
shapes_file = PREFIXES + "\n".join(c["shape"] for c in CONSTRAINTS)
req = urllib.request.Request(
    "http://localhost:3030/ds/shacl?graph=urn:x-arq:UnionGraph",
    data=shapes_file.encode(), headers={"Content-Type": "text/turtle"})
t0 = time.perf_counter()
report = urllib.request.urlopen(req, timeout=1800).read().decode()
shacl_s = time.perf_counter() - t0
shacl_found = report.count("sh:ValidationResult")

print(f"\nSHACL   {shacl_found:>3} violations  {shacl_s:5.2f}s   all four shapes, one request")
print(f"\nCypher: {cypher_found} violations in {cypher_s:.2f}s, four queries written for this graph.")
print(f"SHACL:  {shacl_found} violations in {shacl_s:.2f}s, one file that any triple store would accept.")


--- a start date a human could have meant ---------------------
SHACL
            kg:StartDateIsPlausible a sh:NodeShape ;
                sh:targetClass voc:Contract, voc:ContractRecord ;
                sh:property [ sh:path voc:startDate ;
                              sh:minInclusive "1900-01-01"^^xsd:date ;
                              sh:maxInclusive "2100-12-31"^^xsd:date ] .

  Cypher
            MATCH (n) WHERE (n:Contract OR n:ContractRecord)
              AND n.startDate IS NOT NULL
              AND (n.startDate < date('1900-01-01') OR n.startDate > date('2100-12-31'))
            RETURN n.id AS id, 'startDate' AS path, n.startDate AS value

--- and the other 3, same shape, different path ------------------
   an end date a human could have meant                    voc:endDate              maxInclusive, minInclusive
   an approval date inside the period this dataset covers  voc:approvalDate         maxInclusive, minInclusive
   one contracting authority per record        

Cypher   12 violations   0.20s   a start date a human could have meant
           4558         startDate      0204-01-01
           4957         startDate      0201-07-01
Cypher    3 violations   0.10s   an end date a human could have meant
           148762       endDate        2104-01-24
           149783       endDate        3030-11-30
Cypher    0 violations   0.06s   an approval date inside the period this dataset covers


Cypher    0 violations   0.12s   one contracting authority per record



SHACL    15 violations   9.76s   all four shapes, one request

Cypher: 15 violations in 0.49s, four queries written for this graph.
SHACL:  15 violations in 9.76s, one file that any triple store would accept.


Fifteen, and fifteen. Four hand-written queries and one file agree, which is the
only reason the rest of this comparison means anything.

**The clock is the least interesting column.** The two timings above are not
"Cypher beats SHACL": they are four indexed lookups over 274,646 nodes against
one validation pass over a 36.8 M-triple union graph. Give SHACL an index and the
gap closes; give Cypher the whole graph to scan and it opens the other way.

Two things the clock hides, and both are the point:

**A fifth shape is missing, and so is a fifth Cypher query.** Module 2 wrote
`kg:OneAwardDate` — `sh:maxCount 1` on the award date. In an LPG it cannot fail:
`approvalDate` is a scalar property and the store will not hold two of them. The
constraint is enforced by the data model, so there is nothing to check. In RDF a
subject may carry any number of values for a predicate, so the same sentence has
to be written down and tested. `kg:OneAuthority` is the mirror image: also
`sh:maxCount 1`, and **not** vacuous — the authority is an edge, and an edge can
repeat. The same constraint is empty in one case and alive in the other,
depending on whether the value is a property or a relationship.

**The three engines do not have the same SHACL, and two of them have none.**
Fuseki exposes a `/shacl` endpoint and validates its own graph on request.
GraphDB validates at **commit**, on what you insert — never on what is already
stored — and it ships switched off. Oxigraph does not implement SHACL at all.
What looked like a property of the standard is a property of each engine.

### GraphDB's other answer: refuse the write

Fuseki inspects what is already stored. GraphDB cannot — but it can stand at the
door, and that is a different product with a different price.

Its SHACL runs inside the transaction: the shapes live in a reserved named
graph, every commit is validated against them, and a violation makes the commit
fail. It ships switched **off**, and the switch is not in the repository
configuration file — only in the repository's JSON, over the REST API.

Two repositories below, identical but for that one flag, take the same 822,780
triples through the same reasoner. Note what the data looks like going in: our
own `voc:` names. Nothing in it mentions PCO, and the shapes only speak PCO.

In [24]:
# GraphDB's SHACL, switched on, measured against itself. Two repositories differ
# in ONE setting -- isShacl -- and take the same payload through the same
# reasoner. Both halves of the price are read off the result.
import json, os, sys, time, urllib.error, urllib.parse, urllib.request
sys.path.insert(0, os.path.join("..", "..", "tools", "03-where-data-lives"))

HOST = "http://localhost:7200"
SHAPE_GRAPH = "http://rdf4j.org/schema/rdf4j#SHACLShapeGraph"
VOC, PCO = "http://kgtp.org/voc/", "http://purl.org/procurement/public-contracts#"
BATCH = 25_000
BOUNDARY = "kgtpboundary"

CONFIG = """@prefix rep: <http://www.openrdf.org/config/repository#> .
@prefix sr: <http://www.openrdf.org/config/repository/sail#> .
@prefix sail: <http://www.openrdf.org/config/sail#> .
@prefix graphdb: <http://www.ontotext.com/config/graphdb#> .
[] a rep:Repository ; rep:repositoryID "%(id)s" ; rdfs:label "%(id)s" ;
   rep:repositoryImpl [ rep:repositoryType "graphdb:SailRepository" ;
     sr:sailImpl [ sail:sailType "graphdb:Sail" ;
                   graphdb:ruleset "rdfsplus-optimized" ] ] ."""


# ------------------------------------------------------------------ plumbing
def send(url, data, ctype, method="POST"):
    """One HTTP write. A SHACL refusal arrives as a 500, so it is a status and
    not an exception: the caller counts it."""
    req = urllib.request.Request(url, data=data, method=method,
                                 headers={"Content-Type": ctype})
    try:
        return urllib.request.urlopen(req, timeout=1800).status
    except urllib.error.HTTPError as e:
        e.read()
        return e.code


def ask(repo, sparql):
    q = urllib.parse.urlencode({"query": sparql})
    req = urllib.request.Request(f"{HOST}/repositories/{repo}?{q}",
                                 headers={"Accept": "text/csv"})
    return urllib.request.urlopen(req, timeout=900).read().decode().strip().split("\n")[-1]


def make_repo(repo, shacl):
    """Create the repository, then switch validation on or off.

    The flag cannot go in CONFIG above: GraphDB ignores `graphdb:isShacl` there
    and leaves it false. It is settable only by reading the repository's own
    JSON and PUTting it back with the field changed -- which is also why GraphDB
    ships with validation off and stays that way unless someone goes looking.
    """
    body = (f"--{BOUNDARY}\r\nContent-Disposition: form-data; name=\"config\"; "
            f"filename=\"r.ttl\"\r\nContent-Type: text/turtle\r\n\r\n"
            f"{CONFIG % {'id': repo}}\r\n--{BOUNDARY}--\r\n").encode()
    send(f"{HOST}/rest/repositories", body,
         f"multipart/form-data; boundary={BOUNDARY}")
    cfg = json.load(urllib.request.urlopen(urllib.request.Request(
        f"{HOST}/rest/repositories/{repo}", headers={"Accept": "application/json"})))
    cfg["params"]["isShacl"]["value"] = "true" if shacl else "false"
    send(f"{HOST}/rest/repositories/{repo}", json.dumps(cfg).encode(),
         "application/json", method="PUT")
    return ask(repo, "SELECT ?v WHERE { ?v ?p ?o } LIMIT 0")


def drop_repo(repo):
    send(f"{HOST}/rest/repositories/{repo}", None, "application/json",
         method="DELETE")


# ------------------------------------------------- the payload, with OUR names
# Contract dates exactly as we export them: voc:startDate, never pc:startDate.
# The shapes only ever see the PCO form, and only because the reasoner derives
# it -- so this measures the whole chain, not just the validator.
props = ", ".join(f"<{VOC}{p}>" for p in ("startDate", "endDate", "approvalDate"))
construct = f"""CONSTRUCT {{ ?s ?p ?o . ?s a ?t }} WHERE {{
    ?s a ?t . FILTER (?t IN (<{VOC}Contract>, <{VOC}ContractRecord>))
    ?s ?p ?o . FILTER (?p IN ({props})) }}"""
req = urllib.request.Request(f"{HOST}/repositories/kgtp",
    data=urllib.parse.urlencode({"query": construct}).encode(),
    headers={"Content-Type": "application/x-www-form-urlencoded",
             "Accept": "application/n-triples"})
t0 = time.perf_counter()
payload = urllib.request.urlopen(req, timeout=1800).read().splitlines(keepends=True)
print(f"payload: {len(payload):,} triples in {time.perf_counter() - t0:.1f}s, "
      f"sent in {-(-len(payload) // BATCH)} transactions of {BATCH:,}\n")

from shacl import shapes_from_module2
shapes, _ = shapes_from_module2(include_sparql=False)
alignment = open(os.path.join("vocab", "alignment-voc-pco.ttl"), "rb").read()

# ------------------------------------------------------------------ the A / B
RES = {}
for repo, shacl in (("kgtp-shacl-off", False), ("kgtp-shacl-on", True)):
    make_repo(repo, shacl)
    send(f"{HOST}/repositories/{repo}/statements", alignment, "text/turtle")
    send(f"{HOST}/repositories/{repo}/rdf-graphs/service?graph="
         + urllib.parse.quote(SHAPE_GRAPH, safe=""), shapes.encode(), "text/turtle")

    accepted = refused = 0
    t0 = time.perf_counter()
    for i in range(0, len(payload), BATCH):
        code = send(f"{HOST}/repositories/{repo}/statements",
                    b"".join(payload[i:i + BATCH]), "application/n-triples")
        accepted += code == 204
        refused += code != 204
    elapsed = time.perf_counter() - t0

    inside = int(ask(repo, f"SELECT (COUNT(*) AS ?n) WHERE {{ ?s <{PCO}startDate> ?o }}"))
    RES[shacl] = (elapsed, accepted, refused, inside)
    print(f"isShacl {str(shacl).lower():<5}  {elapsed:6.1f}s   "
          f"transactions accepted {accepted:>3}, refused {refused:>3}   "
          f"pc:startDate inside: {inside:>9,}")
    drop_repo(repo)

# ------------------------------------------------- what the last column costs
off, on = RES[False], RES[True]
lost = off[3] - on[3]
print(f"""
   {100 * (on[0] / off[0] - 1):+.0f}% of time, and {on[2]} of {on[1] + on[2]} transactions refused. The time is the
   half everyone quotes. The LAST COLUMN is the half nobody does.

   SHACL here runs inside the transaction, so a transaction that fails fails
   WHOLE. The {lost:,} pc:startDate facts missing from the validated repository are
   not {lost:,} bad facts: this graph has FIFTEEN impossible dates, and they took
   {lost - 15:,} good ones down with them -- every other fact that happened to be
   travelling in the same batch of {BATCH:,}.

   So validation at the door is not free and it is not precise: it is as coarse
   as your transactions are large. Smaller batches would lose less and cost more,
   and that trade is yours to set -- but only if you know it is there.""")

payload: 822,780 triples in 5.1s, sent in 33 transactions of 25,000



isShacl false    14.7s   transactions accepted  33, refused   0   pc:startDate inside:   190,598


isShacl true     25.9s   transactions accepted  24, refused   9   pc:startDate inside:    95,573



   +75% of time, and 9 of 33 transactions refused. The time is the
   half everyone quotes. The LAST COLUMN is the half nobody does.

   SHACL here runs inside the transaction, so a transaction that fails fails
   WHOLE. The 95,025 pc:startDate facts missing from the validated repository are
   not 95,025 bad facts: this graph has FIFTEEN impossible dates, and they took
   95,010 good ones down with them -- every other fact that happened to be
   travelling in the same batch of 25,000.

   So validation at the door is not free and it is not precise: it is as coarse
   as your transactions are large. Smaller batches would lose less and cost more,
   and that trade is yours to set -- but only if you know it is there.


## What you will be able to say afterwards

- why *the same data in two models* is a sentence to be checked rather than
  assumed, and how you check it: **row counts before timings, every time**
- where an edge property goes when the model has no edges, and what each of the
  three translations costs
- why a set and a list are not the same thing, and how you find out —
  40,632 triples gone, every one of them the same predicate, and no error
  anywhere
- why a benchmark is not a benchmark until the answers match, and why you read
  orders of magnitude from it and ignore the last digits
- which features of module 1 survive a real graph, which survive only as
  estimates, and which do not survive at all
- where the computation should live, with a rule you can apply: the flat cost of
  the export is ruinous for a cheap algorithm and almost free for an expensive one
- that **a constraint can be a file** is the one thing the RDF side has and the
  LPG side does not — and what it is worth once you check it on three engines:
  one validates its own graph on request, one validates only what you insert and
  ships switched off, one does not implement SHACL at all
- and the one that matters most outside this room: **what each engine lets you get
  wrong in silence** — a question answered *false* because RDF is a set, a correct
  query answered *empty* because the graph was not named, a centrality answered
  *zero* because sixteen starting nodes were the default, and a variable-length
  path that means something else here than there

## What this module does not do

It does not name a winner. Every row of the table has a different one, and that
is the answer rather than a way of dodging it.

It does not price the reasoning. You do see inference work: the last cell
validates against `pc:` names and counts 190,598 `pc:startDate` facts, while the
payload it sent in mentions no PCO name at all — every one of those was derived
at load time. What is not here is the bill: what the derivation cost going in,
what it costs on disk, and what it costs on every delete. GraphDB's walk gives
those as figures, 13 axioms and 6,163,208 derived triples, and this session takes
them on trust.

It does not fix the graph. The fifteen impossible dates are still there —
including three contracts that start in the year 0201 — and the entity layer is
still the one the pipeline produced. Recomputed on four of the five engines that
can do it, it comes back at an adjusted Rand index of 0.96 to 0.97 against what
is stored,
which is close and is not the same: a few hundred companies would be grouped
differently depending on which run you asked.

It does not cover distributed engines or cloud costs. Everything here ran on one
laptop, and the numbers are true at that scale and no other.